In [1]:
import os
os.listdir('.')


['engine-spec.md', '.kernel_llm_logs_1.txt', 'memory', '.config', '.prompts']

In [2]:
import mpmath as mp
import numpy as np
import pickle, os, time
mp.mp.dps = 50
# Verify ζ zeros
print(mp.zetazero(1).imag)
print(mp.zetazero(2).imag)


14.134725141734693790457251983562470270784257115699
21.022039638771554992628479593896902777334340524903


In [3]:
# Plan:
# 1. Compute zeros of L(Δ,s) in window [75, 96] (T0=85.7 ± ~10) using mpmath
# 2. Use Z-function approach: completed L-function decays exponentially,
# so use the analytic L(Δ, 1/2 + it) but need careful precision.
# 3. mpmath has lfun? Let's check. There's mp.dirichlet and mp.lerchphi but for Δ
# we'd implement L(Δ,s) = sum tau(n) n^{-s-11/2} (analytic norm: shift by k-1/2=11/2)
# 
# Analytically normalized: L(Δ,s) where critical line is Re(s)=1/2.
# Classical: L_class(s) = sum tau(n)/n^s, critical line Re(s)=6 (k=12 weight)
# Analytic: L_an(s) = L_class(s + 11/2), critical line Re(s)=1/2.
# Functional equation: completed Λ(s) = (2π)^{-s-11/2} Γ(s+11/2) L_class(s+11/2)
# But analytically normalized: Λ(s) = (2π)^{-s-11/2} Γ(s+11/2) L_an(s)
# With Λ(s) = Λ(1-s) (sign +1 for Δ since weight 12, level 1).
# 
# To find zeros on critical line, construct real Z-function:
# Z(t) = e^{iθ(t)} L_an(1/2 + it), where θ(t) makes Z real.
# θ(t) = arg[ (2π)^{-(1/2+it)-11/2} Γ((1/2+it)+11/2) ] = arg[ (2π)^{-it} Γ(6 + it) ]
# Z(t) = (2π)^{-it} ... is the phase. So Z(t) = real-valued function.

# Test computing L(Δ, 1/2 + i*9.222) at first zero
mp.mp.dps = 100
def L_delta_classical(s, nmax=200):
 # sum_{n=1}^{nmax} tau(n) / n^s
 # Use mpmath's ramanujantau? mpmath has mp.taufunction? Let's try
 total = mp.mpc(0)
 for n in range(1, nmax+1):
 tau_n = mp.taufunction(n) if hasattr(mp, 'taufunction') else None
 if tau_n is None:
 break
 total += tau_n / mp.power(n, s)
 return total

# Check mpmath
print(dir(mp))[:20] if False else None
print([x for x in dir(mp) if 'tau' in x.lower() or 'ramanujan' in x.lower()])


['taufrom']


In [4]:
# mpmath doesn't have tau. Let's use sympy.
import sympy
from sympy.ntheory.modular import crt
# sympy has ramanujantau?
print([x for x in dir(sympy) if 'tau' in x.lower() or 'ramanujan' in x.lower()][:20])


[]


In [5]:
from sympy.functions.combinatorial.numbers import partition
# Compute tau via Δ = q ∏(1-q^n)^24, the q-expansion
# tau(n) = coefficient of q^n in q ∏(1-q^n)^24
# Use Euler's pentagonal or direct power series.
# Let's just compute tau values up to some N via the eta product.

def compute_tau(N):
 """tau(n) for n=1..N via Δ(q) = q * (eta(q))^24 where eta_coeff(q) = ∏(1-q^n)"""
 # First compute eta = ∏(1-q^n) up to q^N as a polynomial (Euler's pentagonal numbers).
 # eta = Σ_k (-1)^k q^{k(3k-1)/2}, k in Z.
 import numpy as np
 eta = [0]*(N+1)
 eta[0] = 1
 k = 1
 while True:
 e1 = k*(3*k-1)//2
 e2 = k*(3*k+1)//2
 added = False
 sign = (-1)**k
 if e1 <= N:
 eta[e1] += sign
 added = True
 if e2 <= N:
 eta[e2] += sign
 added = True
 if not added:
 break
 k += 1
 # Now compute eta^24 via repeated squaring/multiplication.
 # Multiply 24 times.
 from math import comb
 def poly_mult(a, b, N):
 c = [0]*(N+1)
 for i in range(N+1):
 if a[i] == 0: continue
 ai = a[i]
 for j in range(N+1-i):
 c[i+j] += ai*b[j]
 return c
 # Repeated squaring: eta^24 = ((eta^2)^2)^2 * eta^... use binary
 # 24 = 16 + 8 = 2^4 + 2^3
 powers = {1: eta}
 cur = eta
 for k in [2,4,8,16]:
 cur = poly_mult(cur, cur, N)
 powers[k] = cur
 eta24 = poly_mult(powers[16], powers[8], N)
 # Δ(q) = q * eta^24, so tau(n) = eta24[n-1]
 tau = [0]*(N+1)
 for n in range(1, N+1):
 tau[n] = eta24[n-1]
 return tau

t0 = time.time()
tau = compute_tau(200)
print("Time:", time.time()-t0)
print("tau(1..10):", tau[1:11])
# Expected: 1, -24, 252, -1472, 4830, -6048, -16744, 84480, -113643, -115920


Time: 0.006865024566650391
tau(1..10): [1, -24, 252, -1472, 4830, -6048, -16744, 84480, -113643, -115920]


In [6]:
# Good. Now compute L_an(1/2 + it) via the approximate functional equation.
# For a self-dual L-function of conductor 1 and Gamma factor Γ_C(s) = (2π)^{-s} Γ(s) at s+11/2:
# We can use a smoothed sum. Standard approach:
# L_an(s) = sum tau(n)/n^{s+11/2} for Re(s)>1 conventional, but on critical line we need
# approximate functional equation with cutoff Q.
#
# Easier: use mpmath's built-in nsum and convergence-acceleration with truncation,
# but on the critical line series diverges absolutely. Use approximate functional equation:
#
# Λ(s) = (2π)^{-s-11/2} Γ(s+11/2) L_an(s) is entire, Λ(s) = Λ(1-s).
# Z(t) = phase * L_an(1/2+it), where Z(t) is real-valued.
#
# Use formula via approximate functional equation:
# L_an(s) = Σ_{n=1}^∞ tau(n) V_s(n) / n^{s+11/2-1/2}? Let me re-derive.
#
# Standard: for L(f,s) = sum a(n)/n^s analytically normalized, with conductor N=1, root number ε=1, gamma factor Γ_C(s+k-1/2) where k=12 means gamma_R for our case:
# Actually for weight k cusp form level 1, Gamma factor is Γ_C(s+(k-1)/2) = (2π)^{-s-(k-1)/2} Γ(s+(k-1)/2).
# So for k=12: Γ_C(s+11/2). N=1, ε=+1.
#
# Approximate functional equation:
# L(s) = Σ a(n) V_+(n/√N)/n^s + ε * Σ a(n) V_-(n*√N)/n^{1-s}
# where V_± involve incomplete gamma.
#
# For self-dual ε=+1, on the critical line s=1/2+it:
# L(1/2+it) = Σ a(n) f(n,t) / n^{1/2+it} + (conjugate-like) Σ a(n) f̄(n,-t)/n^{1/2-it}
#
# Let's use a smoothing function approach. Define
# F(s) = (2π)^{-s-11/2} Γ(s+11/2) L(s)
# F(s) = F(1-s)
# Mellin-Barnes representation:
# L(s) = (1/(2πi)) ∫_{c} G(w) X(s,w) dw, etc.
#
# Simpler: Use Riemann-Siegel-like main sum since coefficients tau(n) grow as n^{11/2} but
# analytic normalization a(n) = tau(n)/n^{11/2} so |a(n)| ≤ d(n).
# 
# Approximate functional equation (Hardy-Littlewood):
# L_an(1/2+it) = Σ_{n≤X} a(n)/n^{1/2+it} * V(n/X) + ε χ(1/2+it) Σ_{n≤Y} a(n)/n^{1/2-it} * V(n/Y)
# with V a smooth cutoff and X*Y = (cond)/((2π)^2) ... this is getting involved.

# Let me just use mpmath's lerchphi-based approach. Actually mpmath has very smart series acceleration via nsum + sumem. On critical line with oscillation, it likely won't work directly.
#
# CLEAN APPROACH: implement Möbius-smoothed sum
# L_an(s) = lim_{Y->∞} Σ_n a(n) e^{-n/Y} / n^s, regularized.
# But that's not exactly right on critical line.
#
# Best: implement using the integral representation:
# Λ(s) = ∫_0^∞ (f(it/(2π)) - δ-correction) t^{s+11/2 - 1} dt / something... 
#
# Actually for cusp forms there's no polar correction (L is entire). We have:
# Λ(s) = (2π)^{-s-11/2} Γ(s+11/2) L(f, s+11/2) [classical norm]
# Set f(τ) = Δ(τ) = Σ tau(n) q^n, q=e^{2πi τ}.
# Λ_classical(s) = ∫_0^∞ f(iy) y^{s-1} dy = Σ tau(n) Γ(s)/(2πn)^s -- standard.
# Cleanly: Λ_cl(s) = ∫_0^∞ f(iy) y^{s-1} dy and split at y=1 + use f(i/y) = y^12 f(iy) (modular).
# 
# Then Λ_cl(s) = ∫_1^∞ f(iy) [y^{s-1} + y^{11-s}] dy. (since k=12)
# This converges rapidly! f(iy) ≈ e^{-2π y} for y large.
# 
# In analytic normalization, s_an = s_cl - 11/2:
# Λ_an(s) := Λ_cl(s+11/2) = ∫_1^∞ f(iy) [y^{s+11/2-1} + y^{11-s-11/2}] dy
# = ∫_1^∞ f(iy) [y^{s+9/2} + y^{9/2-s}] dy
# Symmetric under s -> 1-s ✓ since y^{s+9/2} + y^{9/2-s} = (1-s case gives same).
# Indeed: at 1-s: y^{(1-s)+9/2} + y^{9/2-(1-s)} = y^{11/2-s} + y^{s+7/2}. Hmm wait that doesn't match.
# Let me redo: Λ_an(1-s) = ∫_1^∞ f(iy) [y^{(1-s)+9/2} + y^{9/2-(1-s)}] dy
# = ∫_1^∞ f(iy) [y^{11/2-s} + y^{s+7/2}] dy
# vs Λ_an(s) = ∫_1^∞ f(iy) [y^{s+9/2} + y^{9/2-s}] dy
# These don't match: 11/2-s vs 9/2-s and 7/2+s vs 9/2+s. Error somewhere.
#
# Redo carefully. Standard Hecke: Λ(f,s) := (2π)^{-s} Γ(s) L(f,s) = ∫_0^∞ f(iy) y^{s-1} dy.
# Functional eq: Λ(s) = i^k Λ(k-s). For k=12: Λ(s) = Λ(12-s). Critical line Re(s) = k/2 = 6.
# Split at y=1: Λ(s) = ∫_0^1 + ∫_1^∞ = (substitute y->1/y in first) ∫_1^∞ f(i/y) y^{-s-1} (-)dy
# wait: ∫_0^1 f(iy) y^{s-1} dy. Let u=1/y: y=1/u, dy = -du/u^2. When y=0, u=∞; y=1, u=1.
# = ∫_∞^1 f(i/u) (1/u)^{s-1} (-du/u^2) = ∫_1^∞ f(i/u) u^{-s-1} du.
# f(i/u) = u^k f(iu) i^? Actually Δ has weight 12: Δ(-1/τ) = τ^12 Δ(τ). With τ = i/u: -1/τ = iu. So Δ(iu) = (i/u)^{12} Δ(i/u) = u^{-12} Δ(i/u) (since i^12=1). Thus Δ(i/u) = u^12 Δ(iu).
# = ∫_1^∞ u^{12} f(iu) u^{-s-1} du = ∫_1^∞ f(iu) u^{11-s} du.
# Λ(s) = ∫_1^∞ f(iy) [y^{s-1} + y^{11-s}] dy. ✓
# Symmetric s -> 12-s: y^{12-s-1} + y^{11-(12-s)} = y^{11-s} + y^{s-1} ✓.
#
# In analytic normalization s_an = s_cl - 11/2:
# Λ_an(s_an) = Λ_cl(s_an + 11/2) = ∫_1^∞ f(iy) [y^{s_an+11/2-1} + y^{11-s_an-11/2}] dy
# = ∫_1^∞ f(iy) [y^{s_an + 9/2} + y^{9/2 - s_an}] dy
# Symmetry s_an -> 1-s_an: y^{1-s_an+9/2} + y^{9/2-(1-s_an)} = y^{11/2-s_an} + y^{s_an+7/2}
# Doesn't match. So shift is wrong. Let me redo: critical line cl is s_cl=6, want s_an=1/2.
# s_an = s_cl - 11/2. So s_cl = s_an + 11/2. s_cl=6 -> s_an=1/2 ✓.
# Symmetry s_cl -> 12 - s_cl translates to s_an+11/2 -> 12-(s_an+11/2) = 13/2 - s_an,
# i.e., new s_an' = 13/2-s_an-11/2 = 1 - s_an ✓.
# So Λ_an(s_an) under s_an->1-s_an should equal itself. Let's verify:
# Λ_an(1-s_an) = Λ_cl(1-s_an + 11/2) = Λ_cl(13/2 - s_an).
# But Λ_cl(s_cl) symmetric under s_cl -> 12-s_cl. Is 13/2-s_an = 12 - (s_an+11/2)? Yes: 12-s_an-11/2 = 13/2-s_an. ✓
# Good. So Λ_cl(13/2-s_an) = ∫_1^∞ f(iy) [y^{13/2-s_an - 1} + y^{11-(13/2-s_an)}] dy 
# = ∫_1^∞ f(iy) [y^{11/2-s_an} + y^{s_an+9/2}] dy
# vs Λ_an(s_an) = ∫_1^∞ f(iy) [y^{s_an+9/2} + y^{9/2-s_an}] dy
# These are NOT equal: 11/2-s_an vs 9/2-s_an. So I made an error.
# 
# Re-examine: Λ_cl(s_cl) = ∫_1^∞ f(iy) [y^{s_cl-1} + y^{11-s_cl}] dy.
# Plug s_cl = s_an+11/2:
# y^{s_an+11/2-1} + y^{11-s_an-11/2} = y^{s_an+9/2} + y^{11/2-s_an}. 
# I had a 9/2 typo for the second term. Let me redo: 11 - 11/2 = 11/2. So second exponent = 11/2 - s_an. NOT 9/2 - s_an. My arithmetic was wrong.
# So Λ_an(s_an) = ∫_1^∞ f(iy) [y^{s_an+9/2} + y^{11/2-s_an}] dy.
# Symmetry s_an -> 1-s_an: y^{1-s_an+9/2} + y^{11/2-(1-s_an)} = y^{11/2-s_an} + y^{s_an+9/2}. ✓ Match!
# 
# So Λ_an(s) = ∫_1^∞ Δ(iy) [y^{s+9/2} + y^{11/2-s}] dy.
# Δ(iy) = Σ_{n=1}^∞ tau(n) e^{-2π n y}.
# 
# Substituting: Λ_an(s) = Σ_n tau(n) ∫_1^∞ e^{-2πny} [y^{s+9/2} + y^{11/2-s}] dy
# = Σ_n tau(n) [Γ(s+11/2, 2πn)/(2πn)^{s+11/2} + Γ(13/2-s, 2πn)/(2πn)^{13/2-s}]
# where Γ(a,x) = ∫_x^∞ e^{-t} t^{a-1} dt is upper incomplete gamma, after substitution t=2πny.
# 
# Then L_an(s) = Λ_an(s) / [(2π)^{-s-11/2} Γ(s+11/2)] = (2π)^{s+11/2} Λ_an(s) / Γ(s+11/2)
# 
# This is excellent for computation!
print("OK, derivation done")


OK, derivation done


In [7]:
# Implement L_an(s) and Z(t) for L(Delta)
mp.mp.dps = 100

# Pre-compute tau(n) for needed range. e^{-2π n} for n=200 is ~e^{-1257}, negligible.
# For high precision (100 digits), need n such that e^{-2π n} < 10^{-105}.
# 10^{-105} = e^{-242}. So 2πn > 242 -> n > 38.5. Use Nmax = 60 for safety.
# Actually for t around 90, we have polynomial factors y^{s+9/2}=y^{1/2+it+9/2}=y^{5+it} of magnitude y^5.
# At y=1, magnitude 1. As y grows e^{-2πy} dominates. But each tau(n) integral is over y>=1, with weight e^{-2πny}, polynomial.
# Need Γ(s+11/2, 2πn) which decays as e^{-2πn} (2πn)^{s+9/2}. So nmax to get e^{-2πn}*(2πn)^{6} < 10^{-110}: 2πn ≈ 260, n ≈ 42. Use 60.

Nmax = 80
tau = compute_tau(Nmax)
print("tau computed up to", Nmax)


tau computed up to 80


In [8]:
def L_delta_an(s, Nmax=80):
 """Analytically-normalized L(Δ,s) via approximate functional equation (incomplete Gamma)."""
 s = mp.mpc(s)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 # Λ_an(s) = Σ tau(n) [Γ(s+11/2, 2πn)/(2πn)^{s+11/2} + Γ(13/2-s, 2πn)/(2πn)^{13/2-s}]
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, Nmax+1):
 if tau[n] == 0:
 continue
 x = two_pi * n
 term1 = mp.gammainc(a1, x) / mp.power(x, a1)
 term2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (term1 + term2)
 # L_an(s) = (2π)^{s+11/2} Λ_an(s) / Γ(s+11/2)
 L = mp.power(two_pi, a1) * total / mp.gamma(a1)
 return L

# Test at first zero: γ₁ = 9.2223793999211025
mp.mp.dps = 60
t0 = mp.mpf('9.2223793999211025')
val = L_delta_an(mp.mpc(mp.mpf('0.5'), t0))
print("L_an(1/2 + i*9.2223794):", val)
print("|val|:", abs(val))


L_an(1/2 + i*9.2223794): (0.00000000000000000374091923648370360503709168438849759348973736253808547084002 - 0.0000000000000000223555214300363684098097142157092112417047475679631884076172j)
|val|: 0.0000000000000000226663586652710745983451857792056370101171065719395496794646


In [9]:
# Very small ~10^-17, consistent with first zero. Let's confirm by evaluating at slightly different t.
mp.mp.dps = 60
for tt in ['9.22', '9.222', '9.2223794', '9.23', '10']:
 v = L_delta_an(mp.mpc('0.5', tt))
 print(f"t={tt:>12} |L| = {mp.nstr(abs(v), 6)}")


t= 9.22 |L| = 0.00242388


t= 9.222 |L| = 0.000386589


t= 9.2223794 |L| = 8.03964e-11


t= 9.23 |L| = 0.00777278


t= 10 |L| = 0.843434


In [10]:
# Excellent. Now construct Z(t) = real function on critical line.
# Z(t) = e^{iθ(t)} L_an(1/2 + it)
# θ(t) = -Im[log γ(1/2+it)], where γ(s) = π^{-s/2}? No, we need the gamma factor for L_an.
# Gamma factor: γ_∞(s) = (2π)^{-s-11/2} Γ(s+11/2).
# Λ(s) = γ_∞(s) L(s). 
# For self-dual, Λ(1/2+it) ∈ R when t real? No, Λ(1-s)=Λ(s) means Λ(1/2-it) = Λ(1/2+it). Also Λ(s̄)=Λ(s)̄ (real coefficients). So Λ(1/2+it) = Λ(1/2-it)̄, so Λ(1/2+it) is real (assuming ε=+1).
# Hence Z(t) = Λ(1/2 + it) / |γ_∞(1/2+it)| would also be real, but we want
# Z(t) = L_an(1/2+it) e^{iθ(t)} real means we need to absorb the gamma factor's phase.
# Since Λ(1/2+it) = γ_∞(1/2+it) L_an(1/2+it) is real, we have
# arg(L_an(1/2+it)) = -arg(γ_∞(1/2+it))
# So Z(t) = L_an(1/2+it) * exp(i * arg(γ_∞(1/2+it))) is real-valued.
# 
# Simplest: Z(t) = Λ_an(1/2+it) / |γ_∞(1/2+it)|, but Λ already has the exponential decay.
# We want sign-changes only — so just take Z(t) = Re part after multiplying by phase, OR use Λ directly.
# 
# Since Λ(1/2+it) is real, just scan sign of Λ(1/2+it). But Λ decays exponentially with t.
# Better: scan sign of Λ * |γ_∞|^{-1} = L_an phase-rotated.
# 
# Implementation:
def Lambda_an(t):
 """Λ_an(1/2+it) — real-valued for real t."""
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, Nmax+1):
 if tau[n] == 0:
 continue
 x = two_pi * n
 term1 = mp.gammainc(a1, x) / mp.power(x, a1)
 term2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (term1 + term2)
 return total # = Λ_an(s), should be real

mp.mp.dps = 60
for tt in [5, 9, 9.22, 9.222379, 10, 12]:
 v = Lambda_an(mp.mpf(tt))
 print(f"t={tt:>10} Λ = {mp.nstr(v, 6)}")


t= 5 Λ = (0.000352048 + 0.0j)


t= 9 Λ = (1.56403e-6 + 0.0j)


t= 9.22 Λ = (1.37586e-8 + 0.0j)


t= 9.222379 Λ = (2.30751e-12 + 0.0j)


t= 10 Λ = (-2.10987e-6 + 0.0j)


t= 12 Λ = (-6.03046e-7 + 0.0j)


In [11]:
# Lambda decays. At t=85.7, decay factor is roughly (2π)^{-6} * |Γ(6+it)| which has |Γ(6+it)| ~ e^{-πt/2} t^{5.5}
# At t=85.7: e^{-π*85.7/2} ~ e^{-134.6} ~ 10^{-58.5}. So Λ ~ 10^{-58} * (small L_an oscillation).
# Need at least 100 digits of working precision to find sign changes.
# 
# Let's estimate magnitude at t=85
mp.mp.dps = 200
Nmax = 120
tau = compute_tau(Nmax)
print("recomputed tau up to", Nmax)
v = Lambda_an(mp.mpf('85.7'))
print("Λ_an(0.5+85.7i) =", mp.nstr(v, 10))


recomputed tau up to 120


Λ_an(0.5+85.7i) = (-9.9647162e-53 + 0.0j)


In [12]:
# So at dps=200 and Nmax=120, magnitude is ~10^-53 which is comfortably resolved.
# Need to choose Nmax based on dps. Estimate: error from truncating at Nmax is ~ Γ(13/2,2π*Nmax)/(2π*Nmax)^{13/2-s} ~ e^{-2π Nmax} (Nmax)^5.
# At Nmax=120: e^{-754} ~ 10^{-327}. Plenty.
# But for dps=100 we need at most ~e^{-2π N} < 10^{-110}, so N ~ 42. We'll keep Nmax=80 for safety at dps=100.

# Compute L_an phase-corrected to get real Hardy-Z:
# Hardy Z(t) = Λ_an(1/2+it) / |γ_∞(1/2+it)| (real-valued, modulus relates to |L_an|)
# But sign changes of Z are same as sign changes of Λ_an (since |γ_∞|>0).
# So we can scan sign of Re(Lambda_an).
# 
# Now zero finding strategy: scan t in [75, 96] with step Δt. Zero spacing for L(Δ) near t=85.7?
# Density: zeros up to T have count ~ (T/π) log(T/(2π e)) * 2 (for degree 2 L-function).
# Actually for degree d L-function: N(T) ~ d * T log T / (2π). d=2 here.
# Near T=85.7, spacing ~ 2π/(d log(T/(2π))) ~ 2π/(2 * log(13.6)) ~ π/2.6 ~ 1.2.
# Use step 0.05 to be safe.

mp.mp.dps = 120
Nmax = 80
tau = compute_tau(Nmax)

def Lambda_an_fast(t, Nmax=Nmax):
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, Nmax+1):
 if tau[n] == 0: continue
 x = two_pi * n
 t1 = mp.gammainc(a1, x) / mp.power(x, a1)
 t2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (t1 + t2)
 return total

t0 = time.time()
v = Lambda_an_fast(mp.mpf('85.7'))
print("Λ at 85.7:", mp.nstr(v,10), "time:", time.time()-t0)


Λ at 85.7: (-9.9647162e-53 + 0.0j) time: 2.2342445850372314


In [13]:
# 2.2 sec per evaluation at dps=120 is too slow. Try dps=100 and Nmax=60.
mp.mp.dps = 100
Nmax_local = 60

def Lambda_fast(t, Nmax=Nmax_local):
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, Nmax+1):
 if tau[n] == 0: continue
 x = two_pi * n
 t1 = mp.gammainc(a1, x) / mp.power(x, a1)
 t2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (t1 + t2)
 return total

t0 = time.time()
v = Lambda_fast(mp.mpf('85.7'))
print("Λ at 85.7:", mp.nstr(v,10), "time:", time.time()-t0)


Λ at 85.7: (-9.9647162e-53 + 0.0j) time: 1.3612425327301025


In [14]:
# 1.3s per eval. For step 0.05 over [75, 96], that's 420 evals = ~9 minutes. Acceptable.
# But we can optimize: precompute (2π n) and use cleaner gammainc. Let's also try lower precision adaptively.
# For sign-change detection, dps=60 should suffice (Λ ~ 10^-53, need ~7 digits of relative precision).
# But gamma function evaluations need many digits because of cancellation in Γ(a,x).
# Let's test dps=70.

mp.mp.dps = 70
v70 = Lambda_fast(mp.mpf('85.7'), Nmax=50)
print("dps=70 N=50:", mp.nstr(v70, 12))
mp.mp.dps = 100
v100 = Lambda_fast(mp.mpf('85.7'), Nmax=60)
print("dps=100 N=60:", mp.nstr(v100, 12))


dps=70 N=50: (-9.96471620017e-53 + 0.0j)


dps=100 N=60: (-9.96471620017e-53 + 0.0j)


In [15]:
# Good, dps=70 matches 100 to many digits. Use dps=80 + Nmax=55 for scanning.
mp.mp.dps = 80

def Lambda_scan(t, Nmax=55):
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, Nmax+1):
 if tau[n] == 0: continue
 x = two_pi * n
 t1 = mp.gammainc(a1, x) / mp.power(x, a1)
 t2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (t1 + t2)
 return total.real # real part (imag should be ~0)

t0 = time.time()
v = Lambda_scan(mp.mpf('85.7'))
print("time:", time.time()-t0, "value:", mp.nstr(v, 10))


time: 0.813305139541626 value: -9.9647162e-53


In [16]:
# 0.8s per call. For 420 calls = ~6 minutes. Let's go.
# Actually let me first do coarse scan with step 0.1 (210 calls = ~3 min)
mp.mp.dps = 80
t_lo, t_hi = mp.mpf('75'), mp.mpf('96')
step = mp.mpf('0.1')

import time
t_vals = []
sgn_vals = []
t = t_lo
t0 = time.time()
while t <= t_hi + step/2:
 v = Lambda_scan(t)
 t_vals.append(t)
 sgn_vals.append(v)
 t += step
print("scan done in", time.time()-t0, "sec, total points:", len(t_vals))


scan done in 166.8513126373291 sec, total points: 211


In [17]:
# Find sign changes
sign_changes = []
for i in range(len(sgn_vals)-1):
 if sgn_vals[i] * sgn_vals[i+1] < 0:
 sign_changes.append((t_vals[i], t_vals[i+1], sgn_vals[i], sgn_vals[i+1]))

print(f"Found {len(sign_changes)} sign changes:")
for a,b,va,vb in sign_changes:
 print(f" [{mp.nstr(a,6)}, {mp.nstr(b,6)}] Λ: {mp.nstr(va,3)} -> {mp.nstr(vb,3)}")


Found 18 sign changes:
 [75.7, 75.8] Λ: 9.3e-47 -> -2.99e-47
 [77.1, 77.2] Λ: -1.4e-49 -> 5.65e-48
 [77.6, 77.7] Λ: 3.02e-48 -> -5.65e-49
 [79.7, 79.8] Λ: -2.33e-49 -> 1.38e-50
 [80.5, 80.6] Λ: 3.2e-50 -> -1.93e-50
 [82.0, 82.1] Λ: -4.05e-52 -> 3.93e-51
 [82.8, 82.9] Λ: 6.35e-52 -> -7.77e-52
 [83.9, 84.0] Λ: -3.57e-52 -> 1.05e-52
 [85.4, 85.5] Λ: 4.4e-53 -> -2.29e-53
 [86.7, 86.8] Λ: -5.08e-54 -> 3.4e-54
 [88.0, 88.1] Λ: 7.07e-55 -> -1.93e-55
 [89.0, 89.1] Λ: -4.22e-56 -> 1.25e-55
 [90.4, 90.5] Λ: 8.54e-57 -> -6.37e-57
 [91.1, 91.2] Λ: -7.11e-58 -> 4.6e-57
 [92.4, 92.5] Λ: 6.46e-58 -> -7.51e-58
 [93.7, 93.8] Λ: -1.57e-58 -> 5.99e-59
 [95.1, 95.2] Λ: 5.3e-60 -> -6.31e-60
 [95.6, 95.7] Λ: -1.61e-60 -> 5.0e-60


In [18]:
# Refine each zero with bisection / mpmath findroot
mp.mp.dps = 80
def Lambda_real(t):
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, 55+1):
 if tau[n] == 0: continue
 x = two_pi * n
 t1 = mp.gammainc(a1, x) / mp.power(x, a1)
 t2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (t1 + t2)
 return total.real

zeros_local = []
t0 = time.time()
for a,b,va,vb in sign_changes:
 # Bisect
 z = mp.findroot(Lambda_real, (a, b), solver='anderson', tol=mp.mpf('1e-25'))
 zeros_local.append(z)
 print(f" zero: {mp.nstr(z, 20)}")
print("Refine time:", time.time()-t0)


 zero: 75.775663765701440002


 zero: 77.102417929218368134


 zero: 77.684242047075506216


 zero: 79.794432668167398978


 zero: 80.562360691735638852


...

 zero: 91.11337928712515284


 zero: 92.446208482582671854


 zero: 93.77237673265634009


 zero: 95.145633334798272095


 zero: 95.624394774321140387
Refine time: 83.42448306083679


In [19]:
# Cross-check: 1.12.a.a first few zeros from LMFDB are:
# 9.222379, 13.907549, 17.442777, 19.656513, 22.336103, 24.834646, 27.703087, 28.831683, ...
# At higher heights, let me sanity check density. Degree d=2 cusp form, count near T:
# N(T) ~ (T/π) log(T/(2π e)) for degree 2 (analytic conductor 1).
# Number of zeros in [75, 96]:
# N(96) - N(75) ≈ (96/π)*log(96/(2πe)) - (75/π)*log(75/(2πe))
import math
def N_approx(T):
 return (T/math.pi)*math.log(T/(2*math.pi*math.e))
print("Expected zeros [75,96]:", N_approx(96) - N_approx(75))
# We found 18. Let me check.


Expected zeros [75,96]: 17.433959621884284


In [20]:
# 17.4 expected, found 18. Within rounding. Spacing looks reasonable.
# Now refine each zero to higher precision and verify by checking |Λ| << |Λ'|*δ
# Save the local zero list with high precision
import pickle, os
os.makedirs('cache', exist_ok=True)
zeros_str = [mp.nstr(z, 40) for z in zeros_local]
print(zeros_str)

with open('cache/L_delta_local_zeros_T0_85.7.pkl', 'wb') as f:
 pickle.dump({
 'T0': '85.7', 'sigma': '2', 'window': '[75, 96]',
 'dps_used': 80, 'Nmax': 55,
 'zeros': zeros_str,
 }, f)
print("Cached.")


['75.77566376570144000184302670966407412402', '77.10241792921836813395250341146625564216', '77.6842420470755062159092974086754678736', '79.79443266816739897760377685209833574888', '80.5623606917356388524559393661086205804', '82.00934084164373687236859272692887351866', '82.8449692154798005323914728233069521331', '83.97726579911686435290325774452029523813', '85.46579597972635927012151652204030181013', '86.75987649506966692518560070127112714727', '88.07857988786184873893524489494047372357', '89.02518372375151208449189037984897764786', '90.45726348417919792383770216513966827886', '91.11337928712515284044177891823616364946', '92.44620848258267185426635186996131185705', '93.77237673265634008961856791866596504641', '95.14563333479827209482459548855522691869', '95.62439477432114038708815433981492177777']
Cached.


In [21]:
# Now set up the trace identity calculation.
# Test functions: Hermite-Gauss centered at T0, width σ, basis dimension J=10.
# 
# Hermite-Gauss basis functions in variable t:
# φ_j(t) = (normalization) * H_j((t-T0)/σ) * exp(-(t-T0)^2 / (2 σ^2))
# 
# For Weil explicit formula:
# Σ_ρ h(γ_ρ) = (arithmetic side terms involving log, gamma, primes)
# where h is a test function with Fourier transform g.
# 
# The matrix M is built with (φ_j, φ_k) and we compute matrix elements.
# Specifically M_zeros[j,k] = Σ_ρ φ_j(γ_ρ) φ_k(γ_ρ) (using the symmetric basis on critical line).
# And trace: tr(M_zeros) = Σ_j Σ_ρ φ_j(γ_ρ)^2 = Σ_ρ Σ_j φ_j(γ_ρ)^2 = Σ_ρ K(γ_ρ, γ_ρ)
# where K(t,t') = Σ_j φ_j(t) φ_j(t') is the reproducing kernel of the Hermite-Gauss basis.
# 
# K(t,t) = Σ_j |φ_j(t)|^2. For orthonormal Hermite functions,
# Σ_{j=0}^{J-1} |φ_j(t)|^2 = some kernel.
# 
# Actually the standard explicit formula approach uses test functions f(x) on R with
# Σ_ρ f̂(γ) = arithmetic stuff. The matrix is M[j,k] = ⟨f, basis pairing⟩.
# 
# I don't have a specific formula for tr(M) for L(Δ). Let me think more carefully.
# 
# The trace identity tr(M_zeros) = tr(M_arith) means both sides compute Σ_j (some_pairing_j_j).
# Without a precise formula from the spec, I'll use a standard interpretation:
# tr(M_zeros) = Σ_ρ K(γ_ρ), where K(t) = Σ_{j=0}^{J-1} φ_j(t)^2 (one-variable diag of kernel)
# and equivalently, on the arithmetic side this trace equals an integral involving K against
# the arithmetic measure (prime powers + archimedean + polar).
# 
# Let me write out the Weil explicit formula for L(Δ):
# For a test function h(t) such that g(x) = (1/2π)∫ h(t) e^{-ixt} dt:
# Σ_ρ h(γ_ρ) = (main term polar) + (1/π) ∫ h(t) [Re ψ(6+it) - log(2π)] dt - 2 Σ_n Λ_f(n) g(log n) / n^{1/2}
# where Λ_f(n) = a_p^k log p / n? Specifically for L(f,s) = Π (1 - a_p p^{-s} + p^{-1-2s})^{-1}...
# Actually for analytically normalized L(Δ,s) with Euler product Π (1 - λ_p p^{-s} + p^{-2s})^{-1}
# where λ_p = τ(p)/p^{11/2}, the explicit formula reads:
# Σ_ρ h(γ_ρ) = (1/2π) ∫ h(t) [2 Re ψ(1/4 + it/2) + (1/2)log(N/π^2)]? No wait, gamma factor for Δ is Γ_C, so:
# Gamma_C(s) = (2π)^{-s} Γ(s). Λ(s) = γ_∞(s) L(s) with γ_∞(s) = (2π)^{-s-11/2} Γ(s+11/2) is single Gamma_C(s+11/2).
# Logarithmic derivative: (γ_∞'/γ_∞)(s) = -log(2π) + ψ(s+11/2). At s=1/2+it: -log(2π) + ψ(6+it).
# Weil explicit formula:
# Σ_ρ h(γ) = -2 Σ_n Λ_f(n) g(log n)/√n + (1/2π) ∫ h(t) [(γ_∞'/γ_∞)(1/2+it) + conjugate] dt
# + (residues at poles - but L(Δ) is entire so no polar term)
# So polar term is 0 for L(Δ).
# 
# Now the matrix M_jk: the engine constructs M from test functions. Most natural is:
# M[j,k] is given by paired test functions h_jk(t) = φ_j(t) φ_k(t) (or convolutions).
# Different conventions exist. For trace:
# tr(M_zeros) = Σ_j Σ_ρ φ_j(γ_ρ)^2 = Σ_ρ K_J(γ_ρ)
# where K_J(t) = Σ_{j=0}^{J-1} φ_j(t)^2.
# 
# Then tr(M_arith) should equal:
# (1/2π) ∫ K_J(t) [-2 log(2π) + 2 Re ψ(6+it)] dt - 2 Σ_n Λ_f(n) K̂_J(log n) / √n
# (no polar term).
# 
# Where K̂_J(x) = (1/2π) ∫ K_J(t) e^{-ixt} dt = Σ_j |φ̂_j(x)|^2 type thing.
# 
# Hmm but the spec says trace identity has passed for ζ at 10^-43 residual. So the engine knows. Without re-deriving the exact convention, let me write a consistent one and verify on ζ first.
print("Plan: implement explicit formula for ζ first as a sanity check, then apply to L(Δ).")


Plan: implement explicit formula for ζ first as a sanity check, then apply to L(Δ).


In [22]:
# Let me carefully derive the trace formulation.
# 
# Choose test functions: Hermite-Gauss
# φ_j(t) = (1/√(σ √π 2^j j!)) * H_j((t-T0)/σ) * exp(-(t-T0)^2/(2σ^2))
# These are orthonormal on R: ∫ φ_j(t) φ_k(t) dt = δ_jk.
# 
# Define M[j,k] = sum over zeros pairs as
# M_zeros[j,k] = Σ_ρ φ_j(γ_ρ) φ_k(γ_ρ) (assuming γ_ρ are real on critical line)
# By Weil explicit formula (for self-dual L with no pole):
# Σ_ρ h(γ_ρ) = arith(h) := (1/2π) ∫ h(t) Φ_arch(t) dt - Σ_n Λ_f(n) (g(log n) + g(-log n))/√n
# where Φ_arch(t) = -2 log(2π) + 2 Re ψ(6+it), and g(x) = (1/2π)∫h(t) e^{-ixt} dt (the Fourier transform).
# 
# Setting h_jk(t) = φ_j(t) φ_k(t):
# M_zeros[j,k] = Σ_ρ h_jk(γ_ρ) = arith(h_jk) = M_arith[j,k].
# So trace = Σ_j M[j,j] should match.
# 
# Now h_jj(t) = φ_j(t)^2. The Fourier transform g_jj(x) of φ_j(t)^2:
# Using shift: φ_j(t)^2 = (1/(σ √π 2^j j!)) H_j((t-T0)/σ)^2 exp(-(t-T0)^2/σ^2).
# Let u = (t-T0)/σ: φ_j(t)^2 = (1/(σ √π 2^j j!)) H_j(u)^2 e^{-u^2}.
# 
# g_jj(x) = (1/2π) ∫ φ_j(t)^2 e^{-ixt} dt 
# = (e^{-i x T0}/(2π σ √π 2^j j!)) ∫ H_j(u)^2 e^{-u^2} e^{-i x σ u} σ du
# = (e^{-i x T0}/(2π √π 2^j j!)) ∫ H_j(u)^2 e^{-u^2 - i x σ u} du
# 
# Sum K_J(t) = Σ_j φ_j(t)^2 has nice closed form via Mehler? Actually Σ_j H_j(u)^2 e^{-u^2}/(2^j j!) doesn't telescope easily, but the FT might via Laguerre polynomial as spec hints:
# "use of an analytic Fourier transform for the Hermite-Gauss basis sum, which involves a Laguerre polynomial L_{J-1}^{(1)}"
# 
# Known formula: Σ_{j=0}^{J-1} H_j(u)^2 / (2^j j!) = ?
# Or in Fourier: Σ_{j=0}^{J-1} |φ̂_j(k)|^2 = ?
# 
# Let me find a known identity:
# Σ_{j=0}^{J-1} φ_j(t) φ_j(t') (Christoffel-Darboux) = (J/2)^{1/2} [φ_J(t) φ_{J-1}(t') - φ_{J-1}(t)φ_J(t')] / (t-t')
# But we want diagonal t=t': lim gives Σ |φ_j(t)|^2 = ... involves φ_J, φ_{J-1}, derivatives.
# 
# In Fourier domain Σ |φ̂_j(k)|^2 — since φ̂_j is also Hermite-Gauss (up to i^j and σ→1/σ), but
# Σ |φ̂_j(k)|^2 = Σ |i^j ψ_j(k)|^2 = Σ ψ_j(k)^2 = K_J(k) similar form.
# 
# Actually for the Hermite functions ψ_j(x) (orthonormal, σ=1), the Wigner-like representation
# Σ_{j=0}^{J-1} ψ_j(x)^2 = ? doesn't have super clean form.
# 
# Let's just compute trace numerically.
# 
# tr(M_zeros) = Σ_ρ Σ_j φ_j(γ_ρ)^2 = Σ_ρ K_J(γ_ρ)
# tr(M_arith) = (1/2π) ∫ K_J(t) Φ_arch(t) dt - Σ_n Λ_f(n) [G(log n) + G(-log n)]/√n
# where G(x) = (1/2π) ∫ K_J(t) e^{-ixt} dt.
# 
# Since K_J(t) = Σ φ_j(t)^2 is real and even-ish around T0 (φ_j^2 is positive). So G(x) = (1/2π) ∫ K_J(t) e^{-ixt} dt.
# 
# For Hermite-Gauss centered at T0, φ_j(t) = e^{-(t-T0)^2/(2σ^2)} * H_j((t-T0)/σ) * c_j.
# Let τ = t-T0. Then K_J(t) is a function K̃_J(τ).
# G(x) = (1/2π) e^{-i x T0} ∫ K̃_J(τ) e^{-ixτ} dτ.
# 
# Computing G(x) numerically via brute integration is feasible. Each prime contributes a term.
# 
# Let me just compute everything numerically and see if trace identity holds.

# Set parameters
T0 = mp.mpf('85.7')
sigma = mp.mpf('2')
J = 10

# Load zeros
mp.mp.dps = 60
zeros_real = [mp.mpf(z) for z in zeros_str]
print(f"Loaded {len(zeros_real)} zeros around T0={T0}")
print(f"Range: [{zeros_real[0]}, {zeros_real[-1]}]")


Loaded 18 zeros around T0=85.7
Range: [75.77566376570144000184302670966407412402, 95.62439477432114038708815433981492177777]


In [23]:
# Build Hermite-Gauss orthonormal basis and compute K_J(t)
mp.mp.dps = 60

def hermite_phys(j, x):
 """Physicist's Hermite polynomial H_j(x) via recurrence."""
 if j == 0: return mp.mpf(1)
 if j == 1: return 2*x
 Hm1 = mp.mpf(1)
 H = 2*x
 for k in range(1, j):
 Hp1 = 2*x*H - 2*k*Hm1
 Hm1 = H
 H = Hp1
 return H

def phi(j, t, T0, sigma):
 """Orthonormal Hermite-Gauss function: ∫ φ_j φ_k = δ_jk."""
 u = (t - T0)/sigma
 # norm: 1/sqrt(σ * sqrt(π) * 2^j * j!)
 norm = 1/mp.sqrt(sigma * mp.sqrt(mp.pi) * mp.power(2, j) * mp.factorial(j))
 return norm * hermite_phys(j, u) * mp.exp(-u*u/2)

# Check orthonormality
def quad_phi(j, k):
 f = lambda t: phi(j, t, T0, sigma) * phi(k, t, T0, sigma)
 return mp.quad(f, [T0 - 20*sigma, T0, T0 + 20*sigma])

print("φ_0·φ_0:", quad_phi(0,0))
print("φ_0·φ_1:", quad_phi(0,1))
print("φ_1·φ_1:", quad_phi(1,1))
print("φ_2·φ_3:", quad_phi(2,3))


φ_0·φ_0: 1.0
φ_0·φ_1: 3.70920615068742138573173526154763951336756477875779100245304e-68


φ_1·φ_1: 1.0
φ_2·φ_3: -1.48368246027496855429269410461905580534702591150311640098122e-67


In [24]:
# Orthonormal. Now compute tr(M_zeros) = Σ_ρ K_J(γ_ρ) where K_J(t)= Σ_{j=0}^{J-1} φ_j(t)^2
def K_J(t, J=J, T0=T0, sigma=sigma):
 return sum(phi(j, t, T0, sigma)**2 for j in range(J))

# Test
mp.mp.dps = 60
print("K_J at T0:", K_J(T0))
print("K_J at T0+5σ:", K_J(T0 + 5*sigma))
print("K_J at T0+10σ:", K_J(T0 + 10*sigma))

# Trace from zeros
tr_zeros = sum(K_J(g) for g in zeros_real)
print("tr(M_zeros) =", tr_zeros)


K_J at T0: 0.694217651631028243705644637662669392737796672807166562132371
K_J at T0+5σ: 0.00505670317642398138665374161766161501333477367641457991344553
K_J at T0+10σ: 1.07039285978646409180460564885230629507265703308381922870806e-29
tr(M_zeros) = 8.28445020749061163133034080836721258296959027942166315194668


In [25]:
# For tr(M_arith), we need:
# tr(M_arith) = (1/2π) ∫ K_J(t) Φ_arch(t) dt - Σ_n Λ_f(n) [G(log n) + G(-log n)]/√n
# where Φ_arch(t) = -2 log(2π) + 2 Re ψ(6+it).
# Λ_f(n) = a_n_for_log_p? In the explicit formula for L(f, s):
# log L(f, s) = Σ_n c(n) n^{-s} with c(n) = Λ_f(n)/log n (von Mangoldt-like).
# For self-dual L with Euler factors (1 - α_p p^{-s})(1 - β_p p^{-s}) at unramified p:
# -L'/L (s) = Σ_p Σ_{k≥1} (α_p^k + β_p^k) log p · p^{-ks}
# So Λ_f(p^k) = (α_p^k + β_p^k) log p.
# 
# For analytically normalized L(Δ): Euler factor (1 - λ_p p^{-s} + p^{-2s})^{-1} = (1-α_p p^{-s})(1-β_p p^{-s})
# with α_p + β_p = λ_p = τ(p)/p^{11/2}, α_p β_p = 1.
# 
# Λ_f(p^k) = (α_p^k + β_p^k) log p, computable via recurrence:
# a_0 = 2 (α^0+β^0), a_1 = α+β = λ_p, a_{k+1} = λ_p a_k - a_{k-1}.
# 
# Now Weil explicit formula. Let me be precise:
# Let h(t) be even (or apply to h*even part), Schwartz, with FT
# g(x) = (1/2π) ∫_{-∞}^{∞} h(t) e^{-ixt} dt, so h(t) = ∫ g(x) e^{ixt} dx.
# 
# Σ_ρ h(γ_ρ) = h(i/2)+h(-i/2) (poles, =0 for entire L)
# + (1/2π) ∫ h(t) [ -log(N/π^d) + Σ_v (gamma factor logarithmic derivative) ] dt? 
# 
# Standard form (Iwaniec-Kowalski Theorem 5.12 etc.):
# Σ_ρ h(γ_ρ) = (1/π) ∫ h(t) [Re (Γ'/Γ)(...) ] dt + log(...) term - Σ_n Λ(n) [a(n)+ā(n)] g(log n)/√n
# 
# For Λ_an(s) = (2π)^{-s-11/2} Γ(s+11/2) L_an(s), entire, Λ(s)=Λ(1-s):
# Γ'/Γ at gamma factor pieces gives -log(2π) + ψ(s+11/2). At s=1/2+it: -log(2π) + ψ(6+it).
# 
# Functional eq form: log Λ(1/2+it) + log Λ(1/2-it) = 2 Re log Λ(1/2+it).
# Logarithmic derivative: (Λ'/Λ)(s) = (γ_∞'/γ_∞)(s) + (L'/L)(s).
# 
# By Cauchy/contour over rectangle around critical line, the explicit formula for an entire L:
# Σ_ρ h(γ_ρ) = (1/2π) ∫_{-∞}^{∞} h(t) [(γ_∞'/γ_∞)(1/2+it) + (γ_∞'/γ_∞)(1/2-it)] dt
# - Σ_n Λ_f(n) g(log n) / √n - Σ_n Λ_{f̄}(n) g(-log n)/√n
# For self-dual: Λ_{f̄}(n) = Λ_f(n) so:
# Σ_ρ h(γ_ρ) = (1/2π) ∫ h(t) [-2log(2π) + 2 Re ψ(6+it)] dt - Σ_n Λ_f(n) [g(log n) + g(-log n)]/√n
# 
# Good. Now h(t) = K_J(t), real even-ish (in shifted coordinate).
# 
# Compute archimedean term:
mp.mp.dps = 60
def Phi_arch(t):
 return -2*mp.log(2*mp.pi) + 2*mp.re(mp.digamma(6 + 1j*t))

# Integrate K_J(t) Phi_arch(t) over R, but K_J is essentially supported in T0 ± 8σ
arch_term = mp.quad(lambda t: K_J(t) * Phi_arch(t), 
 [T0 - 12*sigma, T0, T0 + 12*sigma]) / (2*mp.pi)
print("Archimedean term:", arch_term)


Archimedean term: 8.31959165191290678434446801508329343450310318878676478188903


In [26]:
# Now compute arithmetic prime-power term.
# Need G(x) = (1/2π) ∫ K_J(t) e^{-ixt} dt.
# K_J(t) is shifted Hermite-Gauss kernel; FT has nice analytic form.
# K_J(t) = e^{-(t-T0)^2/σ^2} / (σ√π) * Σ_{j=0}^{J-1} H_j(u)^2 / (2^j j!), u=(t-T0)/σ.
# 
# Use Mehler-like formula. Actually for orthonormal Hermite functions ψ_j(u) = H_j(u) e^{-u^2/2}/√(√π 2^j j!):
# Σ_{j=0}^{J-1} ψ_j(u)^2 has closed form via Christoffel-Darboux: 
# Σ_{j=0}^{J-1} ψ_j(u)^2 = J * (ψ_J(u)^2 - ψ_{J-1}(u) ψ_{J+1}(u)) ... actually
# Σ_j ψ_j(u) ψ_j(v) = (J/(...))^{1/2} [ψ_J(u) ψ_{J-1}(v) - ψ_{J-1}(u) ψ_J(v)] / (u-v) (CD).
# 
# For Fourier transform of K_J(t) directly numerically — it has compact-ish support — let's
# just numerically compute G(x) by quadrature for each needed x (log p for p ≤ X).
# 
# But that's expensive. Use the spec hint: analytic FT involves Laguerre L_{J-1}^{(1)}.
# Known identity (Wigner / Heisenberg):
# Σ_{j=0}^{J-1} ψ_j(u) ψ_j(v) e^{i (u+v) k/2}... hmm.
# 
# Actually: For shifted Gaussian, F[e^{-u²} H_j(u)²] is a polynomial in k² times Gaussian.
# Easier closed form using generating functions:
# Σ_{j=0}^∞ z^j ψ_j(u)^2 = (π(1-z²))^{-1/2} exp(-u² (1-z)/(1+z)) [Mehler diagonal]
# Truncate at J-1. Let's just do numerical FT.

# For each integer n with p prime, p^k ≤ X = 10^5, we need G(log n) where n=p^k.
# Compute primes up to 10^5
from sympy import primerange
primes = list(primerange(2, 100001))
print(f"{len(primes)} primes ≤ 10^5")


9592 primes ≤ 10^5


In [27]:
# 9592 primes + a few higher powers. For each, we need G(log(p^k)).
# Direct quadrature of K_J(t) e^{-ix t} for each x is expensive (9592 * quadrature ≈ huge).
# 
# Use analytic FT. K_J(t) e^{-ixt} - the FT of K_J is computable in closed form because each φ_j
# is a Hermite function whose product φ_j(t)^2 = Hermite_2j... has FT in closed form.
# 
# φ_j(t)^2 = (1/(σ √π 2^j j!)) H_j(u)^2 e^{-u^2}, u=(t-T0)/σ.
# 
# H_j(u)^2 can be expanded in even-indexed Hermites. Better: use generating function approach.
# 
# Alternative analytic form: Wigner function W_j(u, k) = ∫ ψ_j(u+y/2) ψ_j(u-y/2) e^{-iky} dy = (-1)^j (1/π) e^{-(u²+k²)} L_j(2(u²+k²))
# where L_j is Laguerre. Setting y=0 and integrating over u with extra shift... hmm.
# 
# Different approach: φ̂_j(k) = ∫ φ_j(t) e^{-ikt} dt = (-i)^j √(2π) (1/√σ) φ̃_j(σ k) e^{-i k T0}
# where φ̃_j is the standard Hermite function with σ=1, T0=0 in variable σk.
# Wait, the FT of Hermite function. If ψ_j(u) = H_j(u)e^{-u²/2}/√(√π 2^j j!), then
# ∫ ψ_j(u) e^{-iku} du = (-i)^j √(2π) ψ_j(k). (Hermite functions are eigenfunctions of FT with eigenvalue (-i)^j)
# 
# φ_j(t) = (1/√σ) ψ_j((t-T0)/σ). So
# ∫ φ_j(t) e^{-ikt} dt = (1/√σ) e^{-ikT0} σ ∫ ψ_j(u) e^{-ikσ u} du = (√σ) e^{-ikT0} (-i)^j √(2π) ψ_j(σk).
# 
# Now G(x) = (1/2π) ∫ K_J(t) e^{-ixt} dt = (1/2π) Σ_j ∫ φ_j(t)^2 e^{-ixt} dt
# Hmm but |φ_j(t)|^2 FT is not just |φ̂_j(x)|^2; it's autocorrelation of φ̂_j.
# 
# Specifically: ∫ φ_j(t)^2 e^{-ixt} dt = (1/2π) (φ̂_j * φ̂_j)(x) [if φ_j real]
# That's convolution... still complicated.
# 
# Let's just do direct numerical FT via mpmath quad. Compact support means manageable.
# Each call ~maybe 100ms. 9592 primes -> 16 minutes. Tight but feasible.
# 
# OR we can vectorize: K_J(t) is smooth on a finite interval; precompute K_J on a dense grid then use FFT-like via quad-prebake.
# 
# Better: K_J(t) e^{-ixt} = e^{-ixT0} K̃_J(τ) e^{-ixτ}, where τ = t-T0.
# K̃_J(τ) has Gaussian decay e^{-τ²/σ²} (since φ_j(t)² ~ e^{-(t-T0)²/σ²} times polynomial).
# So K̃_J(τ) = P_J(τ/σ) e^{-τ²/σ²} for some polynomial P_J of degree 2(J-1) in τ/σ.
# Then ∫ K̃_J(τ) e^{-ixτ} dτ = e^{-σ²x²/4} * Q(x σ) for some polynomial Q.
# 
# So G(x) = (1/2π) e^{-ix T0} e^{-σ²x²/4} * Q(σ x).
# 
# This is a polynomial * Gaussian — analytic and fast!
# 
# Let's compute symbolically.
import sympy as sp
u = sp.Symbol('u')
xi = sp.Symbol('xi') # = σ x, conjugate to u
sig = sp.Symbol('sigma', positive=True)

# Sum_{j=0}^{J-1} ψ_j(u)^2 where ψ_j(u) = H_j(u) e^{-u²/2} / sqrt(sqrt(π) 2^j j!)
J_sym = J
P_u = sum(sp.hermite(j, u)**2 / (sp.sqrt(sp.pi) * 2**j * sp.factorial(j)) for j in range(J_sym))
P_u_simplified = sp.expand(P_u)
print("Polynomial part deg:", sp.Poly(P_u, u).degree())
# K̃_J(τ) = (1/σ) P_u((τ/σ)) * e^{-(τ/σ)²}
# Then ∫_{-∞}^{∞} K̃_J(τ) e^{-ixτ} dτ = (1/σ) ∫ P_u(u) e^{-u²} e^{-i x σ u} σ du 
# = ∫ P_u(u) e^{-u² - i ξ u} du, ξ=xσ
# = e^{-ξ²/4} ∫ P_u(u) e^{-(u + iξ/2)²} du = e^{-ξ²/4} ∫ P_u(v - iξ/2) e^{-v²} dv (contour shift)
# This expands as e^{-ξ²/4} Σ moments of u^k with Gaussian (after shift) → polynomial in iξ times sqrt(π).
# 
# Easiest: ∫ u^k e^{-u² - iξu} du = (-i ∂/∂ξ)^k ∫ e^{-u²-iξu} du = (-i ∂/∂ξ)^k √π e^{-ξ²/4}.

# Compute Q(ξ) = ∫ P_u(u) e^{-u² - iξu} du / e^{-ξ²/4}, where P_u is poly in u.
# Expand P_u in monomials, then for each u^k:
# I_k(ξ) = ∫ u^k e^{-u² - iξu} du.
# Use shift: I_0(ξ) = √π e^{-ξ²/4}, I_k = (-i)^k √π e^{-ξ²/4} H_k(ξ/2)/... wait.
# Known: ∫ u^k e^{-u²-iξu} du = √π e^{-ξ²/4} * (poly in ξ of degree k)
# Actually using Hermite generating function:
# ∫ H_n(u) e^{-u²-iξu} du = √π e^{-ξ²/4} (-iξ)^n
# So expand P_u in Hermite basis, get Q(ξ) directly.

P_in_H = sp.expand(P_u)
print(P_in_H)


Polynomial part deg: 18
4*u**18/(2835*sqrt(pi)) - 2*u**16/(45*sqrt(pi)) + 4*u**14/(7*sqrt(pi)) - 34*u**12/(9*sqrt(pi)) + 83*u**10/(6*sqrt(pi)) - 111*u**8/(4*sqrt(pi)) + 119*u**6/(4*sqrt(pi)) - 105*u**4/(8*sqrt(pi)) + 315*u**2/(64*sqrt(pi)) + 315/(128*sqrt(pi))


In [28]:
# OK let me skip symbolic FT and just do numerical FT via mpmath. 
# K_J supported in T0 ± 10σ approximately, beyond which it's ~e^{-100}.
# For each prime, evaluate ∫_{T0-10σ}^{T0+10σ} K_J(t) e^{-ix t} dt.
# 
# Actually let's compute Q(ξ) using the Hermite expansion formula:
# ∫ H_n(u) e^{-u²-iξu} du = √π e^{-ξ²/4} (-iξ)^n
# So if P_u = Σ c_n H_n(u), then ∫ P_u(u) e^{-u² - iξu} du = √π e^{-ξ²/4} Σ c_n (-iξ)^n
# 
# Expand P_u in Hermite basis:
from sympy.polys.orthopolys import hermite_poly
u = sp.Symbol('u')
P_u = sum(sp.hermite(j, u)**2 / (sp.sqrt(sp.pi) * 2**j * sp.factorial(j)) for j in range(J))
P_u = sp.expand(P_u)
# Express as sum c_n H_n(u). Build Hermite basis up to degree 18.
deg = 18
Hermites = [sp.hermite(n, u) for n in range(deg+1)]
# Convert via solving — but easier: numerical extraction of coefficients via integration:
# c_n = ∫ P_u(u) H_n(u) e^{-u²} du / (√π 2^n n!) (orthogonality)
# 
# Wait, that's the L²(e^{-u²} du) inner product. ∫ H_m H_n e^{-u²} du = √π 2^n n! δ_mn.
# Note P_u already has e^{-u²} factor absorbed? No, P_u = Σ_{j<J} H_j(u)²/(√π 2^j j!) — this is the sum, not yet weighted.
# We want: ∫ P_u(u) e^{-u² - iξu} du.
# Expand P_u(u) = Σ_n c_n H_n(u), where c_n = ∫ P_u(u) H_n(u) e^{-u²} du / (√π 2^n n!).
# Then ∫ P_u(u) e^{-u²-iξu} du = Σ_n c_n √π e^{-ξ²/4} (-iξ)^n = √π e^{-ξ²/4} Σ_n c_n (-iξ)^n.

c = {}
for n in range(deg+1):
 integrand = P_u * sp.hermite(n, u) * sp.exp(-u**2)
 val = sp.integrate(integrand, (u, -sp.oo, sp.oo))
 cn = val / (sp.sqrt(sp.pi) * 2**n * sp.factorial(n))
 cn = sp.simplify(cn)
 c[n] = cn

for n in range(deg+1):
 if c[n] != 0:
 print(f"c_{n} = {c[n]}")


c_0 = 10/sqrt(pi)
c_2 = 45/(2*sqrt(pi))
c_4 = 15/sqrt(pi)
c_6 = 35/(8*sqrt(pi))
c_8 = 21/(32*sqrt(pi))
c_10 = 7/(128*sqrt(pi))
c_12 = 1/(384*sqrt(pi))
c_14 = 1/(14336*sqrt(pi))
c_16 = 1/(1032192*sqrt(pi))
c_18 = 1/(185794560*sqrt(pi))


In [29]:
# Only even Hermites appear (as expected — P_u is even in u).
# Q(ξ) = √π e^{-ξ²/4} Σ_n c_n (-iξ)^n.
# 
# Now G(x) = (1/2π) ∫ K_J(t) e^{-ixt} dt
# K_J(t) = (1/σ) P_u((t-T0)/σ) e^{-(t-T0)²/σ²}
# ∫ K_J(t) e^{-ixt} dt = (1/σ) e^{-ixT0} ∫ P_u(u) e^{-u² - iσx u} σ du = e^{-ixT0} I(σx)
# where I(ξ) = ∫ P_u(u) e^{-u²-iξu} du = √π e^{-ξ²/4} Σ c_n (-iξ)^n.
# 
# So G(x) = (1/2π) e^{-ix T0} √π e^{-σ²x²/4} Σ_n c_n (-iσx)^n.
# 
# For real x: Σ_n c_n (-iσx)^n — only even n contribute (real); these terms give Σ c_{2k} (σx)^{2k} (-1)^k.
# Wait: (-iσx)^n = (-i)^n (σx)^n. For n even = 2k: (-i)^{2k} = (-1)^k.
# So Σ_n c_n (-iσx)^n = Σ_k c_{2k} (-1)^k (σx)^{2k}.
# 
# Thus G(x) = (1/2π) e^{-ixT0} √π e^{-σ²x²/4} Σ_k c_{2k} (-1)^k (σx)^{2k}
# = (1/(2√π)) e^{-ixT0} e^{-σ²x²/4} Σ_k c_{2k} (-1)^k (σx)^{2k} / √π ... wait
# Let me redo factor: √π/(2π) = 1/(2√π).
# G(x) = (1/(2√π)) e^{-ixT0} e^{-σ²x²/4} Σ_k c_{2k} (-1)^k (σx)^{2k}
# 
# But c_{2k} contains 1/√π. So write c_{2k} = c̃_{2k} / √π:
# c̃_0 = 10, c̃_2 = 45/2, c̃_4 = 15, c̃_6 = 35/8, c̃_8=21/32, c̃_10=7/128, c̃_12=1/384, c̃_14=1/14336, c̃_16=1/1032192, c̃_18=1/185794560
# 
# G(x) = (1/(2π)) e^{-ixT0} e^{-σ²x²/4} Σ_k c̃_{2k} (-1)^k (σx)^{2k}

c_tilde = [10, sp.Rational(45,2), 15, sp.Rational(35,8), sp.Rational(21,32), sp.Rational(7,128), sp.Rational(1,384), sp.Rational(1,14336), sp.Rational(1,1032192), sp.Rational(1,185794560)]
c_tilde_mp = [mp.mpf(sp.Rational(c).p) / mp.mpf(sp.Rational(c).q) for c in c_tilde]
print("c̃:", [mp.nstr(c, 6) for c in c_tilde_mp])

def G(x, T0=T0, sigma=sigma):
 """G(x) = (1/(2π)) ∫ K_J(t) e^{-ixt} dt."""
 x = mp.mpf(x)
 sx = sigma * x
 poly = mp.mpf(0)
 sx2 = sx*sx
 p = mp.mpf(1) # (σx)^0
 sign = mp.mpf(1)
 for k in range(len(c_tilde_mp)):
 poly += c_tilde_mp[k] * sign * p
 p *= sx2
 sign = -sign
 gaussian = mp.exp(-sigma*sigma*x*x/4)
 phase = mp.exp(-1j*x*T0)
 return (mp.mpf(1)/(2*mp.pi)) * phase * gaussian * poly

# Test: G(0) should equal (1/(2π)) ∫ K_J(t) dt = (1/(2π)) * J (since each φ_j is orthonormal, ∫ φ_j² = 1)
# So G(0) = J/(2π) = 10/(2π).
print("G(0):", G(0), "expected:", mp.mpf(J)/(2*mp.pi))


c̃: ['10.0', '22.5', '15.0', '4.375', '0.65625', '0.0546875', '0.00260417', '6.97545e-5', '9.68812e-7', '5.38229e-9']
G(0): (1.59154943091895335768883763372514362034459645740456448747667 + 0.0j) expected: 1.59154943091895335768883763372514362034459645740456448747667


In [30]:
# Perfect. Verify with quadrature at some non-zero x
mp.mp.dps = 60
x_test = mp.mpf('1.5')
G_quad = mp.quad(lambda t: K_J(t)*mp.exp(-1j*x_test*t), [T0-12*sigma, T0, T0+12*sigma]) / (2*mp.pi)
G_ana = G(x_test)
print("Quad:", G_quad)
print("Analytic:", G_ana)
print("Diff:", abs(G_quad - G_ana))


Quad: (0.0083504022188382952166767706862993634208277294947787112851804 + 0.00217940401963303212752561651184374382452442933730549445733546j)
Analytic: (0.00835040221883829521667677068629936342082772949477646027773596 + 0.00217940401963303212752561651184374382452442933730490695812961j)
Diff: 2.32641136340371127251910511373665107034693532491337947655483e-51


In [31]:
# Excellent — matches to ~10^-51 (numerical floor of quadrature). Now use this analytic G.
# 
# Prime-power sum: -Σ_n Λ_f(n) [G(log n) + G(-log n)] / √n
# where Λ_f(p^k) = (α_p^k + β_p^k) log p, with λ_p = τ(p)/p^{11/2}, α+β=λ, αβ=1.
# 
# Recurrence: a_k = α^k + β^k. a_0=2, a_1=λ, a_{k+1} = λ a_k - a_{k-1}.
# 
# Cutoff X = 10^5. Sum over all p^k ≤ X.

mp.mp.dps = 60
log_2pi = mp.log(2*mp.pi)

# Note: G(-log n) = conjugate of G(log n)? G(x) = (1/(2π)) e^{-ixT0} e^{-σ²x²/4} polynomial_real(σx).
# G(-x) = (1/(2π)) e^{ixT0} e^{-σ²x²/4} polynomial_real(σx) = conj(G(x)).
# So G(log n) + G(-log n) = 2 Re G(log n).

def G_real(x):
 """Returns 2 Re G(x) for x real, more efficient than computing G twice."""
 x = mp.mpf(x)
 sx = sigma * x
 poly = mp.mpf(0)
 sx2 = sx*sx
 p_pow = mp.mpf(1)
 sign = mp.mpf(1)
 for k in range(len(c_tilde_mp)):
 poly += c_tilde_mp[k] * sign * p_pow
 p_pow *= sx2
 sign = -sign
 gaussian = mp.exp(-sigma*sigma*x*x/4)
 # G(x) + G(-x) = 2 (1/(2π)) cos(x T0) e^{-σ²x²/4} poly = (1/π) cos(x T0) e^{-σ²x²/4} poly
 return mp.cos(x*T0) * gaussian * poly / mp.pi

# Test at x=1.5: G(1.5) + G(-1.5) = 2 Re G(1.5) = 2*0.008350402...
print(G_real(mp.mpf('1.5')))
print(2*G(mp.mpf('1.5')).real)


0.0167008044376765904333535413725987268416554589895529205554719
0.0167008044376765904333535413725987268416554589895529205554719


In [32]:
# Excellent. Now compute the prime-power sum.
# We need tau(p) for all primes p ≤ 10^5. Computing tau via eta-product for n up to 10^5 takes
# O(N^2) which is 10^10 — too slow. We need a fast tau computation.
# 
# Better: compute tau(n) multiplicatively. tau is multiplicative; for prime p, tau(p^k) satisfies:
# tau(p^{k+1}) = tau(p) tau(p^k) - p^{11} tau(p^{k-1}).
# So we only need tau(p) for each prime.
# 
# tau(p) for prime p can be computed by:
# (a) the recurrence on the eta product up to index p (still O(N^2) with pentagonal — actually pentagonal is O(N sqrt(N)) for computing eta^24 up to N).
# Let's compute eta^24 up to 10^5 via faster method.
# 
# Computing eta^24 coefficients up to N=10^5:
# eta has only O(√N) nonzero coefficients (pentagonal numbers). Squaring repeatedly: each multiplication is N×(sparse) = O(N * √N). So eta^24 via O(log 24) ~ 5 squarings of polynomials of length N.
# Wait but after squaring, the polynomial becomes dense. Let me think again.
# 
# eta = sparse (O(√N) terms). eta^2 = also relatively sparse? No, becomes dense quickly.
# 
# eta^24 = Δ where Δ(q) has tau as coefficients. Compute via:
# log eta = -Σ_{n,k} q^{nk}/k = -Σ_m c_m q^m where c_m = sum of 1/k for k | m.
# Then eta^24 = exp(24 log eta) — but easier to just do polynomial squaring 4-5 times with FFT.
# 
# Even better: use sympy or scipy FFT. For N=10^5, FFT multiplication is O(N log N) per multiplication.
# 
# Actually maybe simplest: sage has fast tau but I don't have sage. Use direct sieve method:
# Σ tau(n) q^n = q ∏(1-q^n)^24. Multiply 24 power series; each multiplication by (1-q^n)... wait that's a finite product so direct expansion: ∏(1-q^n) for n=1..N (Euler), then raise to 24th power. The product itself = eta, has sparse expansion.
# 
# eta^24 via FFT: prepare eta as length-N+1 array (mostly zeros, with pentagonal entries). 
# Then eta^24 = eta * eta * ... 24 times via numpy convolution or polynomial multiplication.
# 
# Since eta has only ~2√(2N/3) ~ 260 nonzero entries up to N=10^5, eta^2 has at most O(260²)=68k 
# nonzero entries — still sparse-ish. Hmm.
# 
# Let me just use FFT-based multiplication via numpy.

import numpy as np
def compute_tau_fft(N):
 eta = np.zeros(N+1, dtype=np.float64) # need careful overflow handling
 # tau(n) grows ~ n^{11/2} so for N=10^5, |tau(n)| up to ~10^27. Need int128 or Python ints.
 # Use Python int arithmetic. 
 eta_list = [0]*(N+1)
 eta_list[0] = 1
 k = 1
 while True:
 e1 = k*(3*k-1)//2
 e2 = k*(3*k+1)//2
 sign = (-1)**k
 added = False
 if e1 <= N:
 eta_list[e1] += sign; added = True
 if e2 <= N:
 eta_list[e2] += sign; added = True
 if not added: break
 k += 1
 print("Eta nonzero terms:", sum(1 for x in eta_list if x != 0))
 return eta_list

eta_arr = compute_tau_fft(100000)


Eta nonzero terms: 517


In [33]:
# 517 nonzero terms in eta. eta^2 has up to ~ 517^2 = 267k pairs, but result length N=10^5, with many collisions.
# Actually after the first squaring eta^2 is mostly dense up to its full support.
# 
# Use FFT multiplication. But |tau(n)| up to 10^27 means we need ~90 bits precision, more than float64 (53 bits).
# Options: use numpy with object dtype (Python int) — slow. Or use multiple modular arithmetic (CRT). Or use int128.
# 
# Best approach: compute eta^24 (mod some prime ~2^62) using FFT with numpy int64 wraparound — wait, we need exact tau. We need bignum.
# 
# Alternative: We don't need tau for all n up to 10^5 — we need:
# - tau(p) for all primes p ≤ 10^5 (9592 primes), used to compute λ_p
# - tau(p^k) for prime powers ≤ 10^5 (additional ~190 values for higher powers).
# 
# But we need full tau(n) to compute tau(p) easily. Hmm.
# 
# Actually we could compute tau(p) for each prime p using the Hecke recursion on the eta-product:
# Σ tau(n) q^n = q ∏(1-q^n)^24
# But computing tau(p) for a single p still needs all eta^24 coefficients up to p.
# 
# Approach: compute eta^24 coefficients up to N=10^5 with Python big ints, using FFT in chunks. 
# Or just do brute polynomial multiplication 24 times — eta is sparse so each step:
# new[i+j] += eta[i] * cur[j], total work = #nonzero_eta * #nonzero_cur. Initially #nonzero_eta=517, cur starts as [1, 0, ...] -> after 1 mult, cur = eta (517 nz).
# After 2nd: 517*517 = 267k pair-operations. After 23rd similar each round: 517 * (up to N+1) = 5*10^7 per round * 23 rounds = 10^9 — too slow in Python.
# 
# Better approach: use mpmath's series. Or use FFT with int conversion.
# 
# Actually let me use sympy.ntheory? sympy.ntheory has ramanujan tau? Let me check.
from sympy.ntheory.residue_ntheory import primitive_root
# Check for tau
import sympy.ntheory as ntt
print([x for x in dir(ntt) if 'tau' in x.lower() or 'ramanu' in x.lower()])


[]


In [34]:
# No tau in sympy. Let me try sage-like: use the eta product with careful FFT chunked computation.
# 
# Hybrid: 
# Step 1: eta -> eta^2 using sparse multiplication (517 * 517 ≈ 267k ops, ~0.5s)
# Step 2..24: dense * sparse? But dense.
# 
# Strategy: compute eta^3 by sparse-sparse with results in dict, then keep doubling.
# Actually after squaring, the result becomes dense with up to N+1=10^5+1 coefficients.
# So: eta -> eta^2 (sparse * sparse, fast). eta^2 -> eta^4 (dense * dense via FFT). eta^4 -> eta^8 -> eta^16 -> eta^24 = eta^16 * eta^8.
# 
# Dense FFT with int — use numpy fft on floats, but precision is an issue.
# 
# Alternative: use Python's int and built-in multiplication via Karatsuba — multiplying two polynomials of length N as integers and recovering coefficients. Since |coeff| < 10^60 say, packed into ~200-bit slots. N=10^5 coefficients → integer of 200*10^5 = 2*10^7 bits ≈ 6 million decimal digits. Python int multiplication of two such numbers via GMP is fast (~ms).
# 
# Approach: pack a polynomial as a single big int with each coefficient in a 64*B-bit slot, then multiply, then unpack.

def poly_mul_intpack(a, b, N, slot_bits=256):
 """Multiply two polynomials with integer coefficients up to length N+1 each.
 Returns polynomial (list of ints) truncated to length N+1.
 Assumes |coeff(a)*coeff(b)*length| fits in 2^slot_bits."""
 # Encode a and b as integers
 def encode(p):
 # Handle negatives: shift by offset?
 # All ramanujan tau / coefficients can be negative.
 # Trick: use complementary representation. Or split into pos/neg parts.
 # Simpler: shift each coeff by an offset.
 pass
 # Forget this — let's just use numpy.fft for FFT multiplication with floats.

# Actually: simpler to use numpy.convolve since we only need result length N+1.
# But numpy.convolve with default dtype overflows for large ints.
# Use object dtype:
import numpy as np

def poly_mul_object(a, b, N):
 a = np.array(a, dtype=object)
 b = np.array(b, dtype=object)
 result = np.convolve(a, b)[:N+1]
 return result.tolist()

# Try: eta -> eta^2 via numpy.convolve with object dtype
N = 100000
import time
t0 = time.time()
eta = np.array(eta_arr, dtype=object)
print("eta length:", len(eta))
# eta^2 with object dtype convolution — O(N²) but in Python ops, ~10^10 = too slow
# Skip dense convolution, use sparse multiplication recursively.

# Sparse multiplication that returns sparse (using dict)
def sparse_mul(a, b, N):
 """a, b are dicts {index: coef}. Returns dict for product."""
 out = {}
 for i, ai in a.items():
 for j, bj in b.items():
 k = i+j
 if k > N: continue
 out[k] = out.get(k, 0) + ai*bj
 return out

# Convert eta to dict
eta_dict = {i: v for i, v in enumerate(eta_arr) if v != 0}
print("eta dict size:", len(eta_dict))


eta length: 100001
eta dict size: 517


In [35]:
# Strategy: eta has 517 nonzero coefficients up to N=10^5.
# eta^2 will have at most ~ N nonzero coefficients (dense up to N), but the number of pairwise products is 517*517 ≈ 267k.
# eta^4 from eta^2 * eta^2 would be N*N pairs — too many.
# 
# Alternative: use the original sparse eta to do 24 sequential multiplications. Each step:
# cur = cur * eta. cur becomes increasingly dense, but eta stays sparse (517 terms).
# Cost: each step is 517 * (#nonzero in cur, up to N=10^5+1) = 5*10^7 ops in Python.
# 24 steps = 10^9 ops — way too slow in Python.
# 
# Use numpy with object dtype but smarter — represent cur as numpy array (dense), multiply by sparse eta:
# new = zeros(N+1)
# for i, vi in eta_dict.items():
# new[i:] += vi * cur[:N+1-i]
# This uses numpy's vectorized arithmetic. With object dtype, it's still Python ints but the loop overhead is over 517 iterations not N²!
# So each step: 517 * O(N) = 5*10^7 elementary big-int adds. But numpy vectorizes, so should be fast.

def step_mul_sparse_dense(dense, sparse_dict, N):
 new = np.zeros(N+1, dtype=object)
 new[:] = 0 # ensure Python int 0
 for i, vi in sparse_dict.items():
 end = N+1
 slice_len = end - i
 if slice_len <= 0: continue
 # new[i:end] += vi * dense[:slice_len]
 for j in range(slice_len):
 if dense[j] != 0:
 new[i+j] += vi * dense[j]
 return new

# That's still 517*N = 5*10^7 per step, times 24 ≈ 10^9. Too slow.

# Different approach: skip computing all tau(n). 
# We need tau(p) for each prime p ≤ 10^5. Use modular forms recurrence?
# tau(p) for prime p can be computed via direct sum:
# tau(p) = [q^p] (q ∏(1-q^n)^24) -- still needs full series up to p.
# 
# Alternative: use LMFDB or other source? We have web access.
# 
# Actually, we could just compute eta^24 modulo a large prime, then recover (since |tau(p)| < 2 p^{11/2} via Deligne, for p=10^5: |tau| < 2*10^{27.5}, ~10^28).
# Use two moduli to CRT and recover. Or use one modulus > 2 * 10^28: log2(2e28) ≈ 95 bits. Need 96-bit prime.
# 
# numpy FFT in complex128 has ~15 digits precision; product of two N-length polynomials with bounded coeffs can have round-off. The mantissa after FFT requires N * (max_coef_a) * (max_coef_b) to fit in ~2^52. eta has coeffs ±1, so eta^2 has |coef| ≤ N ≈ 10^5. eta^4 has |coef| ≤ N² = 10^10... eventually overflows.
# 
# Best approach now: precompute using GMP-based Python ints via FFT mod some prime, e.g. NTT.
# Or simpler: just use the Hecke recursion at primes p ≤ 10^5 starting from generating function up to a smaller bound — no that's circular.
# 
# Let me try the dense object-dtype convolution but in a smart way: compute eta^24 in log_2(24)≈5 steps using repeated squaring with truncation to N+1 terms.

# Step 1: eta -> eta^2 via sparse-sparse → dense (up to N+1)
t0 = time.time()
eta2_d = sparse_mul(eta_dict, eta_dict, N)
print("eta^2 dict size:", len(eta2_d), "time:", time.time()-t0)


eta^2 dict size: 54976 time: 0.04313802719116211


In [36]:
# 54k terms in eta^2. Now eta^4: convolving 54k * 54k pairs = 3*10^9 - too much.
# But many will collide in dict. Let me check sparsity of eta^2:
nz2 = sum(1 for v in eta2_d.values() if v != 0)
print("eta^2 actual nonzero:", nz2)
print("Density:", nz2 / (N+1))


eta^2 actual nonzero: 34788
Density: 0.34787652123478763


In [37]:
# 35% density. Going up. eta^4 would be denser. Let me try a different approach using a single FFT in floats.
# 
# Actually, given that we need tau(p) for many primes — let me try numpy's FFT-based convolve with floats. The float result for eta^24 may have rounding error ~10^? Let's analyze:
# Max |tau(n)| ≤ 2 n^{11/2}. For n=10^5: |tau| ≤ 2 * 10^{27.5} ≈ 6*10^27.
# float64 has 53 bits of mantissa ≈ 16 digits. We need to recover an integer of magnitude 10^27 — completely impossible in float64.
# 
# So we need big integer arithmetic. Let's just use Python int arithmetic but smartly.
# 
# IDEA: Use the fact that we only need tau(p) for primes (and tau(p^k) for prime-power n ≤ 10^5).
# Primes ≤ 10^5: 9592 primes. Prime powers ≤ 10^5: total ~9783.
# 
# Compute eta^24 = ∏(1-q^n)^24 directly as q*Δ. Use the dense representation with Python ints.
# Use numpy with dtype=object only at the OUTER loop level (24 multiplications by eta), and convert to/from arrays minimally.
# 
# But each multiplication is 517 * N = 5*10^7 Python operations. 24 multiplications = 1.2*10^9. ~30 minutes Python.
# 
# BETTER: Use the structure: eta is "very sparse" with 517 nonzero indices. Multiplying a dense polynomial by eta is doing 517 shifted-vector additions.
# Use numpy with dtype=object — vectorized big-int operations.

# Cur starts as [1] (representing constant 1). We multiply by eta 24 times.
# Wait — we want q * eta^24. So we want eta^24 first, then shift by 1. Equivalent: start cur = eta (which already is "1*eta^1").

# Each step: cur = cur * eta, truncate to length N+1.
# Use numpy array of objects.

cur = np.zeros(N+1, dtype=object)
for i, v in eta_dict.items():
 cur[i] = v

eta_items = list(eta_dict.items())

# Verify with small step: cur after 1 product with eta = eta^2.
# Then multiply 23 more times for eta^24.
t0 = time.time()
for step in range(23): # 23 more multiplications to get eta^24 from eta
 new = np.zeros(N+1, dtype=object)
 # Initialize new array with Python ints
 for j in range(N+1):
 new[j] = 0
 # For each nonzero in eta, add v * cur shifted by i
 for i, v in eta_items:
 # new[i:N+1] += v * cur[:N+1-i]
 # With object arrays, this should work but slowly
 end = N+1
 sl = end - i
 if sl <= 0: continue
 # Numpy can do this — but * on object arrays might be slow.
 if v == 1:
 new[i:end] += cur[:sl]
 elif v == -1:
 new[i:end] -= cur[:sl]
 else:
 new[i:end] += v * cur[:sl]
 cur = new
 if step % 5 == 0:
 print(f"step {step+1}/24 done at {time.time()-t0:.1f}s, sample cur[1]={cur[1]}")
 
print("Total time:", time.time()-t0)
print("eta^24 [coefficients of Δ-without-q-shift]: first few:", [int(cur[i]) for i in range(10)])


step 1/24 done at 0.3s, sample cur[1]=-2


step 6/24 done at 5.6s, sample cur[1]=-7


step 11/24 done at 23.9s, sample cur[1]=-12


step 16/24 done at 50.1s, sample cur[1]=-17


step 21/24 done at 71.2s, sample cur[1]=-22


Total time: 78.17362475395203
eta^24 [coefficients of Δ-without-q-shift]: first few: [1, -24, 252, -1472, 4830, -6048, -16744, 84480, -113643, -115920]


In [38]:
# Total 78 sec. Now tau(n) = cur[n-1] for n >= 1. (since Δ(q) = q * eta^24)
# Wait: cur represents eta^24, so [q^k] eta^24 = cur[k]. Δ = q * eta^24 so [q^n] Δ = cur[n-1].
# Check: tau(1)=1 should be cur[0]=1. ✓ tau(2)=-24 = cur[1]=-24 ✓.

tau_all = [0] + [int(cur[i]) for i in range(N)] # tau_all[n] for n=1..N=100000
print("tau(1..10):", tau_all[1:11])
print("tau(100):", tau_all[100]) # Expected: -39001550
print("tau(1000):", tau_all[1000]) # Expected: -142109721180


tau(1..10): [1, -24, 252, -1472, 4830, -6048, -16744, 84480, -113643, -115920]
tau(100): 37534859200
tau(1000): -30328412970240000


In [39]:
# Let me verify against known values. From LMFDB / OEIS:
# tau(100) — let me compute it: tau is multiplicative; 100 = 4 * 25 = 2² * 5².
# tau(2) = -24, tau(5) = 4830.
# tau(p²) = tau(p)² - p^11
# tau(4) = tau(2²) = 24² - 2^11 = 576 - 2048 = -1472 ✓ (matches our small calc)
# tau(25) = 4830² - 5^11 = 23328900 - 48828125 = -25499225
# tau(100) = tau(4)*tau(25) = (-1472)*(-25499225) = 37534859200 ✓
# 
# tau(1000) = tau(8 * 125) = tau(8) * tau(125)
# tau(8) = tau(2³) = tau(2) tau(4) - 2^11 tau(2) = -24*-1472 - 2048*-24 = 35328 + 49152 = 84480 ✓
# tau(125) = tau(5³) = tau(5)tau(25) - 5^11 tau(5) = 4830*(-25499225) - 48828125*4830
# = 4830 * (-25499225 - 48828125) = 4830 * (-74327350) = -359001100500
# tau(1000) = 84480 * (-359001100500) = -30329693050240000 hmm doesn't match.
# Let me recompute: 84480 * 359001100500 = ?
print(84480 * 359001100500)
print(-30328412970240000 / 84480)


30328412970240000
-359001100500.0


In [40]:
# Wait, my arithmetic was off. 84480 * 359001100500 = 30328412970240000. ✓ Matches!
# Let me redo: 4830 * 25499225 = ? 4830 * 25000000 = 1.2075e11; 4830 * 499225 = 2.41e9; total ~1.23e11. So tau(25) ~ -2.55e7 * 4830 ~ -1.23e11. Yes I made an arithmetic error before. OK.

# Good, tau values are correct! Now compute the prime-power sum.
mp.mp.dps = 60

X = 100000 # cutoff
primes_arr = list(primerange(2, X+1))
print(f"{len(primes_arr)} primes")

# Total prime-power sum
t0 = time.time()
arith_pp = mp.mpf(0)
total_terms = 0
for p in primes_arr:
 lam_p = mp.mpf(tau_all[p]) / mp.power(p, mp.mpf('5.5'))
 log_p = mp.log(p)
 # a_k = α^k + β^k, a_0=2, a_1=lam_p, a_{k+1} = lam_p * a_k - a_{k-1}
 a_km1 = mp.mpf(2)
 a_k = lam_p
 pk = p
 k = 1
 while pk <= X:
 # contribution: -Λ_f(p^k) * G_real(log(p^k)) / sqrt(p^k)
 Lam = a_k * log_p
 log_pk = k * log_p
 contrib = -Lam * G_real(log_pk) / mp.sqrt(pk)
 arith_pp += contrib
 total_terms += 1
 # advance
 pk_new = pk * p
 if pk_new > X: break
 a_kp1 = lam_p * a_k - a_km1
 a_km1 = a_k
 a_k = a_kp1
 pk = pk_new
 k += 1

print(f"Prime-power sum: {arith_pp}")
print(f"Total terms: {total_terms}, time: {time.time()-t0:.1f}s")


9592 primes


Prime-power sum: -0.0336162436716773304033339607967916997672435136204788950919061
Total terms: 9700, time: 1.3s


In [41]:
# Now total tr(M_arith) = arch_term + arith_pp - 0 (no polar)
tr_arith = arch_term + arith_pp
print(f"tr(M_arith) = {tr_arith}")
print(f"tr(M_zeros) = {tr_zeros}")

residual = abs(tr_zeros - tr_arith)
print(f"\nResidual |tr(M_zeros) - tr(M_arith)| = {residual}")
print(f"Pass gate ≈10⁻¹⁵? {residual < mp.mpf('1e-14')}")


tr(M_arith) = 8.28597540824122945394113405428650173473585967516628588679712
tr(M_zeros) = 8.28445020749061163133034080836721258296959027942166315194668

Residual |tr(M_zeros) - tr(M_arith)| = 0.00152520075061782261079324591928915176626939574462273485044062
Pass gate ≈10⁻¹⁵? False


In [42]:
# Residual is 1.5e-3 — much larger than 1e-15. Need to investigate.
# 
# Possible issues:
# (a) Missing zeros — local window [75,96] has 18 zeros; zeros outside window may contribute through K_J tails. K_J at t=T0±10σ is ~10^{-29}, so a zero far away contributes K_J(γ) * (1) ~ 10^{-29}. That's fine.
# 
# But wait K_J(T0+5σ) = 0.005, K_J(T0+10σ) = 10^{-29}. So we need zeros up to T0 ± 10σ. T0=85.7, 10σ = 20. So window should be [65.7, 105.7] not [75, 96]. The current window [75, 96] = T0 ± 10.3 ≈ T0 ± 5σ. K_J at T0±5σ = 5e-3. Zeros just outside contribute up to 5e-3 each!
# 
# Let me extend the search window. T0 ± 10σ = [65.7, 105.7]. K_J(T0 ± 8σ) = ?

print("K_J at T0+6σ:", K_J(T0 + 6*sigma))
print("K_J at T0+7σ:", K_J(T0 + 7*sigma))
print("K_J at T0+8σ:", K_J(T0 + 8*sigma))
print("K_J at T0+9σ:", K_J(T0 + 9*sigma))
print("K_J at T0+10σ:", K_J(T0 + 10*sigma))


K_J at T0+6σ: 0.00000361574551766899331638433742511720304662461931033376032637019
K_J at T0+7σ: 0.000000000171501398460828511738872826729263402534350235723919115592589
K_J at T0+8σ: 0.000000000000000686562036621876218873154919381102381244708724448469442445892
K_J at T0+9σ: 2.6497559767195321349828052278114732950607611291371318457016e-22
K_J at T0+10σ: 1.07039285978646409180460564885230629507265703308381922870806e-29


In [43]:
# So K_J decays fast. At T0+6σ (t=97.7), K_J = 3.6e-6. At T0+7σ (t=99.7), K_J = 1.7e-10. 
# So zeros in (96, 97.7) could contribute up to a few * 10^{-6} — that explains 10^{-3}? Not quite. Let me check more carefully.
# 
# Spacing ~1, so ~1-2 zeros in (96, 99.7) with K_J values ~10^{-6} to 10^{-10}.
# Total contribution: < 10^{-5}. But residual is 10^{-3}.
# 
# Maybe my arithmetic side has issues or my zero list is wrong. Let me check the zero list density.
# 
# Actually wait — the spec says ζ achieved residual 10^-43. So for ζ they had MUCH better — but they used 5000 zeros (covering large range). For L(Δ) the spec says "zero-finding is particularly challenging because the completed L-function decays exponentially on the critical line, requiring very high working precision to find zeros at moderate heights."
# 
# Let me extend window to [65, 106] = T0 ± 10σ.
# And also possibly missed zeros in the original window. Let me scan with finer step.

# First, check what zeros I might have missed in [75, 96] with step 0.1
# Spacing ~1.2 so 0.1 should be safe but let me check at finer step around suspicious regions.
# Actually let's just extend the search.

mp.mp.dps = 80
def Lambda_scan_v2(t, Nmax=55):
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, Nmax+1):
 if tau[n] == 0: continue
 x = two_pi * n
 t1 = mp.gammainc(a1, x) / mp.power(x, a1)
 t2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (t1 + t2)
 return total.real

# Extend: scan [65, 75] and [96, 110]
t0 = time.time()
extra_vals_lo = []
t = mp.mpf('65')
step = mp.mpf('0.1')
while t < mp.mpf('75.05'):
 extra_vals_lo.append((t, Lambda_scan_v2(t)))
 t += step
extra_vals_hi = []
t = mp.mpf('96')
while t < mp.mpf('110.05'):
 extra_vals_hi.append((t, Lambda_scan_v2(t)))
 t += step
print(f"Extended scan done in {time.time()-t0:.1f}s")

# Identify sign changes in low extension
for arr, label in [(extra_vals_lo, 'low'), (extra_vals_hi, 'high')]:
 print(f"--- {label} ---")
 for i in range(len(arr)-1):
 if arr[i][1] * arr[i+1][1] < 0:
 print(f" sign change in [{mp.nstr(arr[i][0],5)}, {mp.nstr(arr[i+1][0],5)}]")


Extended scan done in 187.4s
--- low ---
 sign change in [66.4, 66.5]
 sign change in [67.9, 68.0]
 sign change in [69.0, 69.1]
 sign change in [71.1, 71.2]
 sign change in [71.7, 71.8]
 sign change in [72.8, 72.9]
 sign change in [74.0, 74.1]
--- high ---
 sign change in [97.3, 97.4]
 sign change in [98.7, 98.8]
 sign change in [99.7, 99.8]
 sign change in [100.2, 100.3]
 sign change in [101.3, 101.4]
 sign change in [103.1, 103.2]
 sign change in [103.8, 103.9]
 sign change in [105.2, 105.3]
 sign change in [106.2, 106.3]
 sign change in [107.4, 107.5]
 sign change in [108.4, 108.5]
 sign change in [109.3, 109.4]


In [44]:
# Refine extended zeros
mp.mp.dps = 80
def Lambda_real_fast(t):
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, 55+1):
 if tau[n] == 0: continue
 x = two_pi * n
 t1 = mp.gammainc(a1, x) / mp.power(x, a1)
 t2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (t1 + t2)
 return total.real

extra_zeros = []
for arr in [extra_vals_lo, extra_vals_hi]:
 for i in range(len(arr)-1):
 if arr[i][1] * arr[i+1][1] < 0:
 z = mp.findroot(Lambda_real_fast, (arr[i][0], arr[i+1][0]), solver='anderson')
 extra_zeros.append(z)

extra_zeros.sort()
print(f"Found {len(extra_zeros)} extra zeros")

# Combine with existing local zeros
all_zeros = sorted(extra_zeros + zeros_real)
print(f"Total zeros in [65, 110]: {len(all_zeros)}")
for z in all_zeros:
 print(f" {mp.nstr(z, 12)}, K_J(z) = {mp.nstr(K_J(z), 6)}")


Found 19 extra zeros
Total zeros in [65, 110]: 37
 66.4947692672, K_J(z) = 1.21148e-26
 67.9386097748, K_J(z) = 1.7395e-21
 69.0433978749, K_J(z) = 6.9355e-18
 71.1146534142, K_J(z) = 5.78774e-12
 71.7475041962, K_J(z) = 2.23723e-10
 72.8140606676, K_J(z) = 6.03149e-8
 74.09582544, K_J(z) = 1.90077e-5
 75.7756637657, K_J(z) = 0.0062722
 77.1024179292, K_J(z) = 0.125014
 77.6842420471, K_J(z) = 0.277443
 79.7944326682, K_J(z) = 0.511745
 80.5623606917, K_J(z) = 0.608424
 82.0093408416, K_J(z) = 0.668749
 82.8449692155, K_J(z) = 0.655064
 83.9772657991, K_J(z) = 0.695287
 85.4657959797, K_J(z) = 0.702877
 86.7598764951, K_J(z) = 0.706547
 88.0785798879, K_J(z) = 0.69499
 89.0251837238, K_J(z) = 0.669315
 90.4572634842, K_J(z) = 0.599776
 91.1133792871, K_J(z) = 0.588086
 92.4462084826, K_J(z) = 0.48595
 93.7723767327, K_J(z) = 0.260637
 95.1456333348, K_J(z) = 0.0220033
 95.6243947743, K_J(z) = 0.00627117
 97.3410408898, K_J(z) = 1.63594e-5
 98.7098040882, K_J(z) = 3.26722e-8
 99.7466489

In [45]:
# Now recompute tr(M_zeros)
tr_zeros_full = sum(K_J(g) for g in all_zeros)
print(f"tr(M_zeros) [full] = {tr_zeros_full}")
print(f"tr(M_arith) = {tr_arith}")
residual2 = abs(tr_zeros_full - tr_arith)
print(f"Residual = {residual2}")
print(f"log10(residual): {mp.log10(residual2)}")


tr(M_zeros) [full] = 8.2844856679643509026412739413741233383798540340555325332509135248728843535845311
tr(M_arith) = 8.2859754082412294539411340542865017347358596751662858867971246086482016907628763
Residual = 0.0014897402768785512998601129123783963560056411107533535462110837753173371783452181
log10(residual): -2.8268894404135110544748812932569597468532364575542376987556784313568470596865184


In [46]:
# Still 1.5e-3. Issue is elsewhere. Let me sanity-check by doing the analogous calculation for ζ
# and see what residual I get with my formulas.
# 
# For ζ: gamma factor γ_∞(s) = π^{-s/2} Γ(s/2). 
# At s=1/2+it: (γ_∞'/γ_∞)(s) = -log(π)/2 + (1/2) ψ(s/2) = -log(π)/2 + (1/2) ψ(1/4 + it/2)
# Sum + conjugate = -log(π) + Re ψ(1/4 + it/2)
# So Φ_arch,ζ(t) = -log(π) + Re ψ(1/4 + it/2)
# 
# Also ζ has POLE at s=1 (and s=0 for completed). Explicit formula has polar contribution:
# h(i/2) + h(-i/2) (from poles of ξ at s=0,1, which is s=1/2 ± i*(±i/2)?)
# Actually the standard convention: write ρ = 1/2 + iγ, so for s=0 pole: ρ corresponds to γ = i/2; for s=1: γ = -i/2.
# So poles contribute h(i/2) + h(-i/2) where here h = K_J.
# 
# K_J evaluated at imaginary argument t = i/2: K_J(i/2) involves φ_j(T0 + i/2 - T0)/σ — at T0=85.7, this is K_J(85.7 + 0.5i) using the analytic continuation.
# Actually no — we want K_J(t) = Σ φ_j(t)^2 evaluated at complex t. At t = i/2 (small imaginary), shifted from T0=85.7 — wait the pole contribution comes from h(i*(1/2)) where the test function is centered at T0=85.7. So h(i/2) ≈ K_J(0.5i) — far from T0, essentially zero by Gaussian decay.
# 
# K_J at t=0.5i means u = (0.5i - 85.7)/2 = -42.85 + 0.25i, huge. exp(-u²/2) = exp(-(42.85² - 0.0625)/2 + i*small) → exp(-918) → astronomically small.
# So polar contribution for ζ is negligible for our T0=85.7.
# 
# Let me just sanity check ζ trace identity with my framework.
# 
# Use mp.zetazero to get zeros of ζ. Need zeros in roughly T0 ± 10σ for ζ near T=85.7.
# Density of ζ zeros at height T: spacing ~ 2π/log(T/(2π)) ~ 2π/log(13.6) ~ 2.4. So ~ 8 zeros in [75, 96].
# Actually density for ζ is N(T) ~ (T/(2π)) log(T/(2π e)).

import math
def N_zeta(T):
 return (T/(2*math.pi))*math.log(T/(2*math.pi*math.e))
print("ζ zeros [75,96]:", N_zeta(96) - N_zeta(75))
print("ζ zeros [65, 110]:", N_zeta(110) - N_zeta(65))


ζ zeros [75,96]: 8.716979810942142
ζ zeros [65, 110]: 18.782384083225914


In [47]:
# Let's compute for ζ to test my framework.
mp.mp.dps = 50

# Get ζ zeros up to height 130 to safely cover T0 ± 15σ
T0 = mp.mpf('85.7'); sigma = mp.mpf('2'); J = 10

zeta_zeros = []
n = 1
while True:
 g = mp.zetazero(n).imag
 if g > 130: break
 zeta_zeros.append(g)
 n += 1
print(f"{len(zeta_zeros)} ζ zeros up to 130")

# Compute K_J for ζ
def K_J_z(t):
 return sum(phi(j, t, T0, sigma)**2 for j in range(J))

tr_zeros_zeta = sum(K_J_z(g) for g in zeta_zeros)
print(f"tr(M_zeros) for ζ: {tr_zeros_zeta}")


43 ζ zeros up to 130
tr(M_zeros) for ζ: 3.8982947980685309871749953482165299190621046029871


In [48]:
# Now ζ arithmetic side:
# Phi_arch_zeta(t) = -log(π) + Re ψ(1/4 + i*t/2)
# Plus polar: K_J(i/2) + K_J(-i/2) for ζ. The K_J is defined as Σ φ_j², which is meromorphic in t. 
# φ_j(t) at complex t still defined via formula.
# 
# Prime-power sum: -Σ_{p^k ≤ X} 2 log p [G(k log p)] / p^{k/2}
# where for ζ: Λ(p^k) = log p, so contribution = -log p [G(k log p) + G(-k log p)] / p^{k/2} = -log p * G_real(k log p) / p^{k/2}
# (no factor 2 here since g(x) + g(-x) = G_real already counts both directions)
# 
# Wait — let me re-derive. The Weil formula for ζ:
# Σ_ρ h(γ) = h(i/2) + h(-i/2) + (1/2π) ∫ h(t) [(γ_∞'/γ_∞)(1/2+it) + conj] dt - Σ_n Λ(n) [g(log n) + g(-log n)]/√n
# For ζ: γ_∞(s) = π^{-s/2} Γ(s/2), so (γ_∞'/γ_∞)(s) = -log(π)/2 + ψ(s/2)/2.
# Sum + conj: -log(π) + Re ψ(s/2) where s=1/2+it → Re ψ(1/4 + it/2).
# Λ(p^k) = log p, so prime-power term = -Σ_p Σ_k log p [G(k log p) + G(-k log p)] / p^{k/2} = -Σ log p G_real(k log p) / p^{k/2}.
# (Here G(x) = (1/2π) ∫ h(t) e^{-ixt} dt, my G function.)

def Phi_arch_zeta(t):
 return -mp.log(mp.pi) + mp.re(mp.digamma(mp.mpf('0.25') + 1j*t/2))

# Arch integral
arch_zeta = mp.quad(lambda t: K_J_z(t) * Phi_arch_zeta(t), 
 [T0 - 12*sigma, T0, T0 + 12*sigma]) / (2*mp.pi)
print("ζ arch term:", arch_zeta)

# Prime power sum for ζ, X=10^5
# Need primes ≤ X
from sympy import primerange
primes_z = list(primerange(2, 100001))
print(f"{len(primes_z)} primes")

arith_zeta = mp.mpf(0)
for p in primes_z:
 log_p = mp.log(p)
 pk = p
 k = 1
 while pk <= 100000:
 arith_zeta -= log_p * G_real(k * log_p) / mp.sqrt(pk)
 pk_new = pk * p
 if pk_new > 100000: break
 pk = pk_new
 k += 1
print("ζ prime-power sum:", arith_zeta)

# Polar term: K_J at t = i/2 and -i/2
def phi_complex(j, t, T0, sigma):
 u = (t - T0)/sigma
 norm = 1/mp.sqrt(sigma * mp.sqrt(mp.pi) * mp.power(2, j) * mp.factorial(j))
 return norm * hermite_phys(j, u) * mp.exp(-u*u/2)

def K_J_complex(t):
 return sum(phi_complex(j, t, T0, sigma)**2 for j in range(J))

polar_zeta = K_J_complex(mp.mpc(0, mp.mpf('0.5'))) + K_J_complex(mp.mpc(0, -mp.mpf('0.5')))
print("ζ polar term:", polar_zeta)

tr_arith_zeta = arch_zeta + arith_zeta + polar_zeta
print(f"\ntr(M_arith) for ζ: {tr_arith_zeta}")
print(f"tr(M_zeros) for ζ: {tr_zeros_zeta}")
print(f"Residual: {abs(tr_zeros_zeta - tr_arith_zeta)}")


ζ arch term: 4.1564980228415609429349584014735431666579385616804
9592 primes


ζ prime-power sum: -0.25820322477302995575996305325701324759583411137367
ζ polar term: (-5.9120560648166326808841806851963403307139600377251e-772 + 0.0j)

tr(M_arith) for ζ: (3.8982947980685309871749953482165299190621044503067 + 0.0j)
tr(M_zeros) for ζ: 3.8982947980685309871749953482165299190621046029871
Residual: 1.5268038510036757147869412138530536670712501759032e-43


In [49]:
# Excellent! For ζ, residual is 1.5e-43 — matches the spec's claim of "~10⁻⁴³"!
# So my framework is correct. The issue must be with L(Δ) — let me debug.
# 
# Differences:
# - L(Δ) gamma factor: γ_∞(s) = (2π)^{-s-11/2} Γ(s+11/2). Φ_arch = -2 log(2π) + 2 Re ψ(6+it). ✓
# - L(Δ) prime term: λ_p = τ(p)/p^{11/2}, Λ_f(p^k) = a_k log p with recurrence. ✓
# - L(Δ) no pole, polar term = 0. ✓
# - Need to find ALL zeros up to where K_J is negligibly small.
# 
# I had 37 zeros in [65, 110]. Check expected count:
# Degree 2: N_Delta(T) ~ (T/π) log(T/(2π e)).
print("Expected L(Δ) zeros [65,110]:", N_approx(110) - N_approx(65))
print(f"Found: {len(all_zeros)}")


Expected L(Δ) zeros [65,110]: 37.56476816645183
Found: 37


In [50]:
# 37 found vs 37.5 expected. Reasonable. Maybe missing one zero. The fact that residual is 10^-3
# suggests something else is off.
# 
# Let me verify my analytic-form Lambda is correct by computing L_an at standard points.
# First check: L_an(s) at s=1: should be a positive real number (L(Δ, 1) classically corresponds to L_an(1) by the shift).
mp.mp.dps = 50

# Actually let's check: L(Δ, s) classical at s=12 is finite, etc.
# We know first zero is at t = 9.222379... I verified this.
# Let me compute L_an(0.5 + 0i) (real point on critical line).
val_half = L_delta_an(mp.mpc('0.5', '0'), Nmax=60)
print("L_an(1/2):", val_half)
# Should be real positive number.


L_an(1/2): (0.79212283864603056935594489048673583815191079427891 + 0.0j)


In [51]:
# Real positive ✓. Look up LMFDB: L(Δ, 1/2) ≈ 0.792...?
# Actually L(Delta, s) special value at s=1/2 (analytic normalization) — LMFDB lists L(1/2) ≈ 0.7921...
# Let me check: LMFDB says L_an(1/2) of 1.12.a.a = 0.7921228... Yes! ✓
# 
# Great, my L_an is correct. So the issue might be:
# (a) Missing zeros
# (b) Incorrect Λ_f recurrence?
# (c) Sign of contribution?
# 
# Let me verify with a higher cutoff or test that my analysis recovers Σ ρ K_J(γ) using a simple test function.
# 
# Actually, let me check if there's a missing zero by looking at sign-change density: I had 18 in [75,96] (vs ~17.4 expected), 7 in [65,75] (vs ~6.3), 12 in [96,110] (vs ~13.9). Could be missing 1-2 zeros in [96,110].
# 
# Let me re-scan [96, 110] with finer step.
mp.mp.dps = 80
new_vals = []
t = mp.mpf('96')
step = mp.mpf('0.05')
while t < mp.mpf('110.01'):
 new_vals.append((t, Lambda_real_fast(t)))
 t += step
print(f"{len(new_vals)} points scanned")

sc = []
for i in range(len(new_vals)-1):
 if new_vals[i][1] * new_vals[i+1][1] < 0:
 sc.append((new_vals[i][0], new_vals[i+1][0]))
print(f"Sign changes ({len(sc)}):")
for a,b in sc:
 print(f" [{mp.nstr(a,6)}, {mp.nstr(b,6)}]")


281 points scanned
Sign changes (12):
 [97.3, 97.35]
 [98.7, 98.75]
 [99.7, 99.75]
 [100.2, 100.25]
 [101.3, 101.35]
 [103.15, 103.2]
 [103.8, 103.85]
 [105.2, 105.25]
 [106.25, 106.3]
 [107.4, 107.45]
 [108.45, 108.5]
 [109.35, 109.4]


In [52]:
# 12 sign changes in [96, 110], same as before. Maybe scan even further.
# Let me also check that K_J is well-defined at far-away points where Hermite polynomial values get huge.
# Also re-scan low side and extend further.

t = mp.mpf('55')
step = mp.mpf('0.1')
low_vals = []
while t < mp.mpf('65.05'):
 low_vals.append((t, Lambda_real_fast(t)))
 t += step
print("low extension:")
for i in range(len(low_vals)-1):
 if low_vals[i][1] * low_vals[i+1][1] < 0:
 print(f" [{mp.nstr(low_vals[i][0],5)}, {mp.nstr(low_vals[i+1][0],5)}]")

t = mp.mpf('110')
hi_vals = []
while t < mp.mpf('120.05'):
 hi_vals.append((t, Lambda_real_fast(t)))
 t += step
print("hi extension:")
for i in range(len(hi_vals)-1):
 if hi_vals[i][1] * hi_vals[i+1][1] < 0:
 print(f" [{mp.nstr(hi_vals[i][0],5)}, {mp.nstr(hi_vals[i+1][0],5)}]")


low extension:
 [55.2, 55.3]
 [56.7, 56.8]
 [58.5, 58.6]
 [59.7, 59.8]
 [61.1, 61.2]
 [62.6, 62.7]
 [64.0, 64.1]
 [64.8, 64.9]


hi extension:
 [110.7, 110.8]
 [111.5, 111.6]
 [112.7, 112.8]
 [113.8, 113.9]
 [115.0, 115.1]
 [116.4, 116.5]
 [117.1, 117.2]
 [118.1, 118.2]
 [119.0, 119.1]
 [119.9, 120.0]


In [53]:
# OK let me think about the residual more carefully. Even if I add zeros way out at t=120, K_J ≈ 0.
# For t=120, K_J ~ 10^? Let me check.
print("K_J(110):", K_J(mp.mpf('110')))
print("K_J(120):", K_J(mp.mpf('120')))
print("K_J(65):", K_J(mp.mpf('65')))
print("K_J(55):", K_J(mp.mpf('55')))


K_J(110): 8.2433690108805895327210522996057850247479600642022475757618765749063692521003544e-49
K_J(120): 1.0811899750540312331227915641468580213623638097973063040613847217058911793794511e-109
K_J(65): 1.6399852196813916658630050193071598738186324924157279194943161418466929793792347e-32
K_J(55): 3.6434845982006808441315320568158820443223275444541986225591276470717097495242291e-85


In [54]:
# K_J at 110 is 8e-49 — adding zeros up there contributes maybe 5*8e-49 = 4e-48 to trace. Not 1e-3.
# At t=65, K_J = 1.6e-32 — adding zeros there contributes ~1e-31. Still tiny.
# 
# So the 10^-3 residual is NOT from missing zeros. There must be something wrong with the arithmetic side calculation.
# 
# Let me sanity check: my analytic G is based on a fixed sigma = 2 and T0 = 85.7. The integration of Phi_arch with K_J integrates K_J(t) over t — but I derived G(x) assuming the same K_J. That's consistent.
# 
# Test: take h = K_J. Compute Σ_ρ h(γ_ρ) for L(Δ). Equal to arch_term - prime_term?
# That's what I've been doing.
# 
# Let me try X_cutoff much larger or check whether dropping the cutoff matters.
# At X = 10^5, the tail Σ_{p > 10^5} contributes? Each term is O(λ_p log p * G_real(log p) / √p).
# |λ_p| ≤ 2 (Deligne). |G_real(log p)| at p=10^5: log p ≈ 11.5, σ * x = 23. e^{-23²/4} = e^{-132} ~ 10^{-58}.
# So tail is negligible.
# 
# Let me check whether I have the SIGN right. In ζ test: Λ(p^k) = log p, term -log p * G_real(...)/√p.
# I got 1.5e-43 residual for ζ. So sign was right for ζ.
# For L(Δ): same structure, just Λ_f(p^k) = a_k * log p where a_k can be negative. Let me check a few terms.

# Check small primes
mp.mp.dps = 50
print("τ(2) =", tau_all[2], "λ_2 =", mp.mpf(tau_all[2])/mp.power(2, mp.mpf('5.5')))
print("τ(3) =", tau_all[3])
print("τ(5) =", tau_all[5])
# Ramanujan-Petersson: |λ_p| ≤ 2
for p in [2,3,5,7,11,13,17,19,23,29]:
 lam = mp.mpf(tau_all[p])/mp.power(p, mp.mpf('5.5'))
 print(f"p={p}: λ_p = {mp.nstr(lam, 6)}, |λ| = {mp.nstr(abs(lam), 6)}")


τ(2) = -24 λ_2 = -0.53033008588991064330063327157863677946362695326636
τ(3) = 252
τ(5) = 4830
p=2: λ_p = -0.53033, |λ| = 0.53033
p=3: λ_p = 0.598734, |λ| = 0.598734
p=5: λ_p = 0.691213, |λ| = 0.691213
p=7: λ_p = -0.376548, |λ| = 0.376548
p=11: λ_p = 1.00087, |λ| = 1.00087
p=13: λ_p = -0.431561, |λ| = 0.431561
p=17: λ_p = -1.17965, |λ| = 1.17965
p=19: λ_p = 0.987803, |λ| = 0.987803
p=23: λ_p = 0.603975, |λ| = 0.603975
p=29: λ_p = 1.16251, |λ| = 1.16251


In [55]:
# λ values look correct (all in [-2, 2]).
# 
# Let me very carefully compute and isolate the issue.
# 
# It's possible my G(x) formula needs more terms — particularly for the LARGE x like log p ≈ 11.5 for p=10^5.
# σ * x = 23, then (σx)^{2k} for k=0..9 = up to (23)^{18} = 10^{24.5}, times c_{18}/√π ≈ 5e-9, gives ~10^{15} pre-Gaussian. Then e^{-σ²x²/4} = e^{-132} ~ 10^{-57}. Product ~10^{-42}.
# 
# But (σx)^{2k} doesn't converge well — the series is alternating with growing terms before Gaussian damping. At σx=23 and k=9, term is c̃_{18} * (-1)^9 * 23^{18}/√π = (5.4e-9) * (-1) * 7.4e23 / 1.77 = -2.3e15.
# This is OK because the Gaussian kills it: 2.3e15 * e^{-132} ~ 10^{-42}.
# 
# But cancellation! Series with alternating O(10^15) terms cancelling to give result O(10^{-42} * something) — needs ~57 digits of precision in arithmetic. At dps=60 this might be marginal.
# 
# Wait, the polynomial in (σx)² IS exact — Σ c̃_{2k} (-1)^k (σx)^{2k} doesn't cancel within itself; it's evaluated, then multiplied by tiny gaussian. Let me check.
# 
# Hmm actually each term has alternating sign! Let me check magnitudes for x = log(10^5) = 11.5:
sigma = mp.mpf('2')
xval = mp.log(100000)
sx2 = (sigma * xval)**2
print("σx =", sigma*xval)
print("(σx)² =", sx2)
poly_val = mp.mpf(0)
p = mp.mpf(1)
sign = mp.mpf(1)
for k in range(len(c_tilde_mp)):
 term = c_tilde_mp[k] * sign * p
 print(f" k={k}: c_tilde={c_tilde_mp[k]}, sign={sign}, (σx)^{2*k}={p}, term={term}")
 poly_val += term
 p *= sx2
 sign = -sign
print("poly sum:", poly_val)
print("gaussian:", mp.exp(-sigma*sigma*xval*xval/4))
print("product:", poly_val * mp.exp(-sigma*sigma*xval*xval/4))
print("G_real(x=log 1e5):", G_real(xval))


σx = 23.025850929940456840179914546843642076011014886288
(σx)² = 530.18981104783980105616663885940176915974500791328
 k=0: c_tilde=10.0, sign=1.0, (σx)^0=1.0, term=10.0
 k=1: c_tilde=22.5, sign=-1.0, (σx)^2=530.18981104783980105616663885940176915974500791328, term=-11929.270748576395523763749374336539806094262678049
 k=2: c_tilde=15.0, sign=1.0, (σx)^4=281101.23573894407116278182784783656178264165565954, term=4216518.536084161067441727417717548426739624834893
 k=3: c_tilde=4.375, sign=-1.0, (σx)^6=149037011.06174502962375953144611715815537064396568, term=-652036923.39513450460394795007676256692974656734984
 k=4: c_tilde=0.65625, sign=1.0, (σx)^8=79017904733.961407545610055993794873326409700426779, term=51855499981.662173701806599245927885620456365905074
 k=5: c_tilde=0.0546875, sign=-1.0, (σx)^10=41894487980295.204790248811264330428371287476696204, term=-2291104811422.3940119667318660180703015547838818237
 k=6: c_tilde=0.0026041666666666666666666666666666666666666666666667, sign=1.0, (

In [56]:
# Hmm, my G_real(x) is computed as: cos(x*T0) * gaussian * poly / pi
# G_real includes a cos(x T0) factor! Let me check:
# Direct product (poly * gaussian / pi) = -3.4e-42 / π = -1.08e-42 ≈ G_real value if cos(x T0)=1.

# Actually for the prime sum we need x = k * log p, T0 = 85.7. cos(x*T0) gives oscillatory factor.
# That's fine.
# 
# The cancellation issue: my polynomial sum at σx=23 has terms up to 10^16 cancelling to give 10^16. That requires no precision loss because it's a finite polynomial evaluated at a single point — not really cancellation.
# 
# Actually wait, look at the terms: they're up to 6e15 but alternating, and final result is -1.25e16. So no cancellation — the leading term k=8 dominates (6e15), with k=9 subtracting 1.8e16... actually the terms grow. Let me plot:
# k=4: +5.2e10
# k=5: -2.3e12
# k=6: +5.8e13
# k=7: -8.2e14
# k=8: +6.0e15
# k=9: -1.8e16
# Each grows by factor ~30. Series isn't converging — but it's a finite polynomial sum (J=10 → max k=9). So result is dominated by k=9.
# 
# Now cancellation: actual cancellation is between k=9 (-1.8e16) and earlier (+6e15). Result -1.25e16, magnitude ~10^16. So we lost about 0.5 decimal digits, no big precision loss.
# 
# OK so G_real is being computed accurately. Let me cross-check with quadrature:

# G_real(x) = 2 * Re G(x) = (1/π) ∫ K_J(t) cos(xt) dt (from definition)
mp.mp.dps = 60
x_test = mp.log(100000)
G_quad = mp.quad(lambda t: K_J(t)*mp.cos(x_test*t), [T0-12*sigma, T0, T0+12*sigma]) / mp.pi
print("Quad G_real(log 1e5):", G_quad)
print("Analytic:", G_real(x_test))


Quad G_real(log 1e5): -1.0660799544517059056780601334181642727872843419699098914813e-42
Analytic: -1.06607919391654476123949276396319030177245825393128404206184e-42


In [57]:
# Match to 6 digits (quadrature isn't perfect at this oscillation level). Analytic is more accurate.
# 
# So everything looks right. Let me directly compute the difference per prime — maybe the issue is 
# with my Λ_f recurrence. Let me carefully re-derive.
# 
# L(f, s) = Π_p (1 - λ_p p^{-s} + p^{-2s})^{-1} for L(Δ) analytically normalized.
# log L = -Σ_p log(1 - λ_p p^{-s} + p^{-2s}) = Σ_p Σ_{k≥1} (λ_p p^{-s} - p^{-2s})^k / k? No, use other expansion.
# 
# Factor: 1 - λ_p X + X² = (1 - α X)(1 - β X) with X = p^{-s}.
# log L = -Σ_p [log(1-α X) + log(1-β X)] = Σ_p Σ_k (α^k + β^k) X^k / k = Σ_p Σ_k (α^k+β^k) p^{-ks}/k
# 
# -L'/L (s) = Σ_p Σ_k (α^k + β^k) log p * p^{-ks}
# 
# So Λ_f(p^k) = (α^k + β^k) log p, where a_k = α^k + β^k.
# a_0 = 2, a_1 = α+β = λ_p, a_2 = α²+β² = (α+β)² - 2αβ = λ_p² - 2.
# Check recurrence: a_{k+1} = (α+β)(α^k+β^k) - αβ(α^{k-1}+β^{k-1}) = λ_p a_k - a_{k-1}. ✓
# 
# For Λ_f(p) = a_1 log p = λ_p log p.
# For Λ_f(p²) = a_2 log p = (λ_p² - 2) log p.
# 
# Let me check my code: starts with a_km1 = 2, a_k = lam_p (so at k=1).
# Then computes contribution with Lam = a_k * log_p = λ_p log p ✓ for k=1.
# Then advances: a_kp1 = lam_p * a_k - a_km1 = λ_p² - 2 ✓ for k=2.
# OK the recurrence is correct.
# 
# Let me check the SIGN in the explicit formula one more time.
# Standard reference (Iwaniec-Kowalski):
# For self-dual L with completed Λ(s) = γ_∞(s) L(s), Λ(s)=Λ(1-s):
# Σ_ρ h(γ_ρ) = (1/2π) ∫ h(t) [(γ_∞'/γ_∞)(1/2+it) + (γ_∞'/γ_∞)(1/2-it)] dt 
# - 2 Σ_n (Λ_f(n)/√n) g(log n) + ... wait, for self-dual the two terms combine.
# 
# I had: Σ_ρ h(γ_ρ) = arch_term - Σ_n Λ_f(n) [g(log n) + g(-log n)]/√n.
# 
# So tr_zeros = arch + (- prime sum). ⟹ arch = tr_zeros + prime_sum.
# So tr_zeros = arch - prime_sum. ← this is what I have for L(Δ).
# Same for ζ which gave residual 10^-43.
# 
# Let me check: maybe Lambda_f(p^k) needs an extra factor I missed. Some references use (Λ_f(n) + Λ_{f̄}(n))/√n which for self-dual is 2 Λ_f(n)/√n.
# 
# In Iwaniec-Kowalski 5.12: explicit formula reads
# Σ_ρ h(γ) = ∫ h(r) Σ_{v∈S} ... dr - Σ_{n>=2} Λ(n)/√n [a(n) g(log n) + ā(n) g(-log n)]
# where L(s) = Σ a(n)/n^s and Λ(n) = log p for n=p^k, Λ_f(n) = a(n)? 
# 
# Hmm different conventions. Let me look at the Riemann-Weil formula for L(f,s) more carefully:
# 
# Theorem (Mestre, Weil): For cusp form f of weight k=2k_0, level N:
# Σ_ρ h(γ_ρ) = h(0) (sort of, for newforms?) - 2 Σ_n c(n) g(log n) + (1/2π) ∫ h(t) Ψ_∞(t) dt
# where:
# - c(n) = a_f(n) Λ(n) / (n^{1/2} log n) something like that... 
# 
# OK let me JUST sanity check by computing tr_arith for L(Δ) using a different formulation:
# Compute -L'/L(s) for s=1/2+it as a function. The integral
# tr_arith_via_logderiv = (1/2πi) ∫_{Re(s)=1/2} h(t) [(L'/L)(s) + (γ_∞'/γ_∞)(s) - polar] ds 
# 
# Equivalent to: this contour integral gives Σ residues (zeros). For positive s direction (counter-clockwise around critical strip), but...
# 
# Simpler test: I should be able to recover tr_zeros by computing L'/L on critical line and integrating against h(t):
# Σ_ρ h(γ_ρ) is related to (1/(2πi)) ∫ h(...) (L'/L) ds along boundary of critical strip.
# 
# Let me sanity-check my Λ_f computation. Compute -L'/L(s) at some s and compare with direct Euler product log derivative:
# -L'/L(s) = Σ_p Σ_k a_k log p / p^{ks} (analytically normalized: p^{ks} not p^{k(s+11/2)})
# 
# Test at s=2 (well in convergence region):
# Direct computation: L(s) = Σ tau(n)/n^{s+11/2}. L'(s)/L(s) = ?
# Use Euler product: log L = Σ_p [-log(1 - λ_p p^{-s} + p^{-2s})]
# -L'/L = Σ_p [(-λ_p log p p^{-s} + 2 log p p^{-2s}) / (1 - λ_p p^{-s} + p^{-2s})]

mp.mp.dps = 50
s = mp.mpf('2')
# Compute via formal series with Λ_f
val_series = mp.mpf(0)
for p in primes_arr[:1000]: # first 1000 primes
 lam_p = mp.mpf(tau_all[p])/mp.power(p, mp.mpf('5.5'))
 a_km1 = mp.mpf(2)
 a_k = lam_p
 pk = p
 while pk <= 100000:
 val_series += a_k * mp.log(p) / mp.power(pk, s)
 pk_new = pk * p
 if pk_new > 100000: break
 a_kp1 = lam_p * a_k - a_km1
 a_km1 = a_k
 a_k = a_kp1
 pk = pk_new

# Compare with direct (using mpmath autodiff of L_an or sum tau(n) log n / n^{s+11/2})
val_direct = mp.mpf(0)
for n in range(2, 5001):
 # tau(n) needed for n up to 5000
 if n <= N:
 val_direct += mp.mpf(tau_all[n]) * mp.log(n) / mp.power(n, s + mp.mpf('5.5'))
# Wait the right formula: -L'/L = (-1)*(L'/L).
# L = Σ tau(n)/n^{s'}, s' = s+11/2.
# L' = -Σ tau(n) log(n)/n^{s'}.
# -L'/L = Σ tau(n) log n / n^{s'} / L.

L_val = sum(mp.mpf(tau_all[n])/mp.power(n, s + mp.mpf('5.5')) for n in range(1, 5001))
Lp_val = -sum(mp.mpf(tau_all[n])*mp.log(n)/mp.power(n, s + mp.mpf('5.5')) for n in range(1, 5001))
val_direct = -Lp_val / L_val

print(f"From Λ_f recurrence sum: {val_series}")
print(f"From direct L'/L (truncated): {val_direct}")


From Λ_f recurrence sum: -0.058327226981465113945236196643812393632997041161689
From direct L'/L (truncated): -0.058327035769774628439385188393123670323910606773291


In [58]:
# Match to ~7 digits — the rest is truncation error from n≤5000 in direct sum.
# So my Λ_f recurrence is CORRECT.
# 
# Maybe my issue is precision in the L(Δ) zeros — they're only ~25 digits accurate vs ζ zeros at 50 dps.
# Try refining zeros with more precision.
mp.mp.dps = 100
Nmax = 60
tau = compute_tau(Nmax)

def Lambda_real_hp(t):
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, Nmax+1):
 if tau[n] == 0: continue
 x = two_pi * n
 t1 = mp.gammainc(a1, x) / mp.power(x, a1)
 t2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (t1 + t2)
 return total.real

# Refine a zero to high precision
z = zeros_real[8] # at ~85.5
print("Before refine:", z)
z_hp = mp.findroot(Lambda_real_hp, z)
print("After refine:", z_hp)
print("Lambda at refined:", Lambda_real_hp(z_hp))


Before refine: 85.46579597972635927012151652204030181012999999999999999999999518690179224621868328610684034962868856


After refine: 85.462218148580062049165190400434509578316526267419931496330567337842297118048491644399884309369829


Lambda at refined: -8.638187517077808314401450292290690116894904777348956184569598019549858505542552627601332429549588383e-107


In [59]:
# Oops — the zero shifted by 0.004! That means my zeros at dps=80 might have been WRONG (precision issue).
# Let me re-find all zeros at higher precision.
# 
# Wait actually findroot may have moved due to local minima or insufficient bracket — let me check by 
# evaluating Lambda at both points.
mp.mp.dps = 100
print("Λ at 85.4658:", Lambda_real_hp(mp.mpf('85.4658')))
print("Λ at 85.462:", Lambda_real_hp(mp.mpf('85.462')))
print("Λ at 85.4622181:", Lambda_real_hp(mp.mpf('85.4622181')))


Λ at 85.4658: -2.294329041879867717539059180412006431572347950530306179907417251405709470030537855008676184787095778e-54


Λ at 85.462: 1.405751138342331107579376820442124328256021678707094766664684606386882100843727679148221192216216109e-55


Λ at 85.4622181: 3.129426530945995173876199925206807086260679113858406643171998225946619640368054261871137226911754668e-59


In [60]:
# So actually true zero is at 85.4622, not 85.46579 as I had!
# 
# Wait but at dps=80 with Nmax=55 my Lambda_scan said sign change in [85.4, 85.5].
# Let me check Lambda at dps=80 vs dps=100 with same Nmax.
mp.mp.dps = 80
Nmax_test = 55
tau_test = compute_tau(Nmax_test)
def Ltest(t):
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, Nmax_test+1):
 if tau_test[n] == 0: continue
 x = two_pi * n
 t1 = mp.gammainc(a1, x) / mp.power(x, a1)
 t2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau_test[n] * (t1 + t2)
 return total.real

print("dps=80, Nmax=55:")
print(" at 85.4:", Ltest(mp.mpf('85.4')))
print(" at 85.4622:", Ltest(mp.mpf('85.4622')))
print(" at 85.4658:", Ltest(mp.mpf('85.4658')))
print(" at 85.5:", Ltest(mp.mpf('85.5')))


dps=80, Nmax=55:


 at 85.4: 4.4049888542116301348043204873684671515778547997959156587229643655508927424584321e-53


 at 85.4622: 1.1691271427089311782533513023757990695064255333131462562676198068846644060323078e-56


 at 85.4658: -2.294329041879867717539059180412013004213937651727697611655912986493179243062658e-54


 at 85.5: -2.2899321311307278388635338572161939234802232778780417370316084668977990661103675e-53


In [61]:
# With dps=80, Nmax=55: sign goes +4.4e-53 (at 85.4) → +1.2e-56 (at 85.4622) → -2.3e-54 (at 85.4658) → -2.3e-53 (at 85.5).
# So the zero is between 85.4622 and 85.4658! Both my scans should have detected it. Let me look at my actual sign change record:
# Originally: [85.4, 85.5] Λ: 4.4e-53 -> -2.29e-53. That matches my new values.
# 
# But after refining with mp.findroot solver='anderson' I got 85.46579597972635927012151652204030181012
# Let me check Λ at that value at dps=80:
mp.mp.dps = 80
print("Λ at 85.46579597... (my 'zero'):", Ltest(mp.mpf('85.46579597972635927012151652204030181012')))
print("Λ at 85.462218... (true zero):", Ltest(mp.mpf('85.462218148580062049165190400434509578316526267419931496330567337842297118048491644399884309369829')))


Λ at 85.46579597... (my 'zero'): -2.2917684775386161478880563104276999509806325235895658630342449266650459800293622e-54


Λ at 85.462218... (true zero): -1.0528597541593351891814706953635260205223636296588696343577947642469139323595264e-86


In [62]:
# SO MY ZEROS ARE WRONG. findroot's anderson solver didn't actually find the zero.
# Let me check what happened. mpmath findroot with bracketing should give correct answer.
# Maybe 'anderson' doesn't bracket properly. Let me use 'bisect' or 'secant'.

mp.mp.dps = 80
print("Trying bisect:")
z_bisect = mp.findroot(Ltest, (mp.mpf('85.4'), mp.mpf('85.5')), solver='bisect', tol=mp.mpf('1e-50'))
print("Bisect zero:", z_bisect, "Λ =", Ltest(z_bisect))

print("\nTrying anderson:")
z_anderson = mp.findroot(Ltest, (mp.mpf('85.4'), mp.mpf('85.5')), solver='anderson')
print("Anderson zero:", z_anderson, "Λ =", Ltest(z_anderson))


Trying bisect:


Bisect zero: 85.462218148580062049165190400434527465275176519511782968868923542060667131181617 Λ = -1.1533850098548022570556804228995543792831405944479002504147851521218533956698178e-83

Trying anderson:


Anderson zero: 85.462218148580062049165190400434509633681014146364575871420957690080960335492227 Λ = -4.3923729753612144120131913752366926997231617598930075654343755597572723271714262e-86


In [63]:
# OK so anderson when given a tuple (a,b) does bracket! I previously called findroot with (a,b) tuple too.
# Let me re-examine — my original code:
# z = mp.findroot(Lambda_real, (a, b), solver='anderson', tol=mp.mpf('1e-25'))
# Where a, b were from sign change brackets.
# 
# Wait — both anderson calls give the right answer 85.46221 now! But my earlier zero was 85.46579!
# Let me recheck by re-running the EXACT prior code:

mp.mp.dps = 80
def Lambda_real_OLD(t):
 s = mp.mpc(mp.mpf('0.5'), t)
 two_pi = 2*mp.pi
 total = mp.mpc(0)
 a1 = s + mp.mpf('5.5')
 a2 = mp.mpf('6.5') - s
 for n in range(1, 55+1):
 if tau[n] == 0: continue
 x = two_pi * n
 t1 = mp.gammainc(a1, x) / mp.power(x, a1)
 t2 = mp.gammainc(a2, x) / mp.power(x, a2)
 total += tau[n] * (t1 + t2)
 return total.real

# tau was at Nmax=60 (recomputed at dps=100). At dps=80 should still be valid since computed with int arith.
print("Tau computed at dps=", mp.mp.dps, "via compute_tau which is pure int")
print("tau[1..5]:", tau[1:6])

# Test sign change
print(Lambda_real_OLD(mp.mpf('85.4')))
print(Lambda_real_OLD(mp.mpf('85.5')))

# Re-find
z_new = mp.findroot(Lambda_real_OLD, (mp.mpf('85.4'), mp.mpf('85.5')), solver='anderson')
print("Re-found zero:", z_new)


Tau computed at dps= 80 via compute_tau which is pure int
tau[1..5]: [1, -24, 252, -1472, 4830]


4.4049888542116301348043204873684671515778547997959156587229643655508927424584321e-53


-2.2899321311307278388635338572161939234802232778780417370316084668977990661103675e-53


Re-found zero: 85.462218148580062049165190400434509633681014146364575871420957690080960335492227


In [64]:
# Now I get 85.4622 as the zero. So earlier my output (85.46579) was WRONG.
# Let me re-find all zeros to high precision now and recompute.
mp.mp.dps = 80

# All sign change brackets — combine all 3 scans
all_brackets = []
# Original scan [75, 96] step 0.1
t_vals_orig = []
t = mp.mpf('75')
step = mp.mpf('0.1')
while t <= mp.mpf('96.05'):
 t_vals_orig.append(t)
 t += step
# Combine all and extend
all_t = []
t = mp.mpf('55')
while t <= mp.mpf('120.05'):
 all_t.append(t)
 t += step

t0 = time.time()
all_lam = [Lambda_real_OLD(t) for t in all_t]
print(f"Scanned {len(all_t)} pts in {time.time()-t0:.1f}s")

brackets = []
for i in range(len(all_t)-1):
 if all_lam[i] * all_lam[i+1] < 0:
 brackets.append((all_t[i], all_t[i+1]))
print(f"{len(brackets)} sign changes detected")


Scanned 651 pts in 496.0s
55 sign changes detected


In [65]:
# Refine all zeros
mp.mp.dps = 80
t0 = time.time()
zeros_refined = []
for a, b in brackets:
 z = mp.findroot(Lambda_real_OLD, (a, b), solver='anderson')
 zeros_refined.append(z)
print(f"Refined {len(zeros_refined)} zeros in {time.time()-t0:.1f}s")
print("First few:", [mp.nstr(z, 10) for z in zeros_refined[:5]])
print("Around T0=85.7:")
for z in zeros_refined:
 if mp.mpf('80') < z < mp.mpf('92'):
 print(f" {mp.nstr(z, 12)}")


TimeoutError: Code execution timed out after 345.0 seconds